In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import sweetviz as sv
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import warnings
from scipy.stats import zscore, mode
from sklearn.metrics import (silhouette_samples,silhouette_score,make_scorer,mean_absolute_error, r2_score, mean_squared_error,accuracy_score,precision_score,recall_score,f1_score,roc_auc_score)
from sklearn.base import (BaseEstimator,TransformerMixin,ClassifierMixin,RegressorMixin)
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, BaseCrossValidator, KFold, cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
import joblib
from joblib import load
import os
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.clustering import TimeSeriesKMeans, silhouette_score as ts_silhouette_score
import shap
from lime import lime_tabular
from sklearn.model_selection import GroupKFold

In [2]:
# Cargar los datos desde el archivo Excel
file_path = 'C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Dataset/dataset_completo.xlsx'
df = pd.read_excel(file_path, sheet_name='dataset')

In [3]:
df.head()

,Empresa,Fecha,P_E,P_B,P_TB,P_S,P_CF,P_FCF,P_Share,ROCE_sp,...,Efectivo y equiv_l,CPI,CPI_Exp_mediana,CPI_Exp_promedio,Fed Funds Rate,Fed Funds Rate_Exp_mediana,Fed Funds Rate_Exp_promedio,Non farm payrolls,Non farm payrolls_Exp_mediana,Non farm payrolls_Exp_promedio
0,FLWS US Equity,20140930,31.7207,2.5643,6.1415,0.6062,10.3190,21.6093,7.19,5.886400,...,1.314000,1.7,0.019,0.0191,0.25,0.0025,0.0025,307,230k,226.13k
1,FLWS US Equity,20141031,35.4266,2.8638,6.859,0.677,11.5246,24.1339,8.03,8.881198,...,2.610333,1.7,0.016,0.0162,0.25,0.0025,0.0025,240,215k,216.22k
2,FLWS US Equity,20141128,37.7207,3.0493,7.3032,0.7208,12.2709,25.6967,8.55,9.469145,...,3.906667,1.3,0.016,0.0157,0.25,0.0025,0.0025,284,235k,236.62k
3,FLWS US Equity,20141231,11.3711,2.3843,8.1913,0.5153,2.6214,3.0449,8.24,8.721900,...,5.203000,0.8,0.014,0.0142,0.25,0.0025,0.0025,278,230k,229.16k
4,FLWS US Equity,20150130,10.8881,2.283,7.8433,0.4934,2.5100,2.9155,7.89,7.711122,...,4.188000,-0.1,0.007,0.0069,0.25,0.0025,0.0025,196,240k,234.73k


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166440 entries, 0 to 166439
Data columns (total 42 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Empresa                         166440 non-null  object 
 1   Fecha                           166440 non-null  int64  
 2   P_E                             87792 non-null   float64
 3   P_B                             119348 non-null  object 
 4   P_TB                            92514 non-null   object 
 5   P_S                             117703 non-null  object 
 6   P_CF                            95713 non-null   float64
 7   P_FCF                           69291 non-null   object 
 8   P_Share                         131345 non-null  object 
 9   ROCE_sp                         115613 non-null  float64
 10  ROCE_l                          115613 non-null  float64
 11  EBIT_sp                         115176 non-null  float64
 12  EBIT_l          

In [3]:
# Quitar "k" de las columnas "Non farm payrolls_Exp_mediana" y "Non farm payrolls_Exp_promedio"
df['Non farm payrolls_Exp_mediana'] = df['Non farm payrolls_Exp_mediana'].astype(str).str.replace('k', '')
df['Non farm payrolls_Exp_promedio'] = df['Non farm payrolls_Exp_promedio'].astype(str).str.replace('k', '')

# Convertir las columnas a valores numéricos
df['Non farm payrolls_Exp_mediana'] = pd.to_numeric(df['Non farm payrolls_Exp_mediana'], errors='coerce')
df['Non farm payrolls_Exp_promedio'] = pd.to_numeric(df['Non farm payrolls_Exp_promedio'], errors='coerce')
df['P_B'] = pd.to_numeric(df['P_B'], errors='coerce')
df['P_TB'] = pd.to_numeric(df['P_TB'], errors='coerce')
df['P_S'] = pd.to_numeric(df['P_S'], errors='coerce')
df['P_FCF'] = pd.to_numeric(df['P_FCF'], errors='coerce')
df['P_Share'] = pd.to_numeric(df['P_Share'], errors='coerce')

In [4]:
# Convertir la columna de fecha a formato datetime
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y%m%d')

# Extraer características temporales relevantes
df['mes'] = df['Fecha'].dt.month
df['año'] = df['Fecha'].dt.year
df['periodo'] = df['Fecha'].dt.to_period('M')

In [5]:
# Convertir CPI y Fed Funds Rate a decimales
df['CPI'] = df['CPI'] / 100
df['Fed Funds Rate'] = df['Fed Funds Rate'] / 100

In [6]:
# Calcular diferencias en puntos básicos
df['dif_CPI_mediana'] = (df['CPI'] - df['CPI_Exp_mediana']) * 10000
df['dif_CPI_promedio'] = (df['CPI'] - df['CPI_Exp_promedio']) * 10000

df['dif_FFR_mediana'] = (df['Fed Funds Rate'] - df['Fed Funds Rate_Exp_mediana']) * 10000
df['dif_FFR_promedio'] = (df['Fed Funds Rate'] - df['Fed Funds Rate_Exp_promedio']) * 10000

# Calcular diferencia
df['dif_NFP_mediana'] = (df['Non farm payrolls'] - df['Non farm payrolls_Exp_mediana'])
df['dif_NFP_promedio'] = (df['Non farm payrolls'] - df['Non farm payrolls_Exp_promedio'])

In [7]:
# Lista de columnas numéricas para análisis con mensualización lineal de los ratios 
# y variables macroeconómicas calculadas con mediana
columnas_numericas_lineal_mediana= [
    'P_E',
    'P_B',
    'P_S',
    'P_Share',
    'ROCE_l',
    'EBIT_l',
    'Total Activos_l',
    'Deuda a LP_l',
    'ROA_l',
    'Beneficio neto_l',
    'ROI_l',
    'EV_l',
    'Cap de mercado_l',
    'Deuda a CP_l',
    'Efectivo y equiv_l',
    'dif_CPI_mediana',
    'dif_FFR_mediana',
    'dif_NFP_mediana'
]

In [8]:
# Lista de columnas numéricas para análisis con mensualización spline de los ratios 
# y variables macroeconómicas calculadas con mediana
columnas_numericas_spline_mediana = [
    'P_E',
    'P_B',
    'P_S',
    'P_Share',
    'ROCE_sp',
    'EBIT_sp',
    'Total Activos_sp',
    'Deuda a LP_sp',
    'ROA_sp',
    'Beneficio neto_sp',
    'ROI_sp',
    'EV_sp',
    'Cap de mercado_sp',
    'Deuda a CP_sp',
    'Efectivo y equiv_sp',
    'dif_CPI_mediana',
    'dif_FFR_mediana',
    'dif_NFP_mediana'
]

In [9]:
# Lista de columnas numéricas para análisis con mensualización lineal de los ratios 
# y variables macroeconómicas calculadas con promedio
columnas_numericas_lineal_promedio = [
    'P_E',
    'P_B',
    'P_S',
    'P_Share',
    'ROCE_l',
    'EBIT_l',
    'Total Activos_l',
    'Deuda a LP_l',
    'ROA_l',
    'Beneficio neto_l',
    'ROI_l',
    'EV_l',
    'Cap de mercado_l',
    'Deuda a CP_l',
    'Efectivo y equiv_l',
    'dif_CPI_promedio',
    'dif_FFR_promedio',
    'dif_NFP_promedio'
]

In [10]:
# Lista de columnas numéricas para análisis con mensualización spline de los ratios 
# y variables macroeconómicas calculadas con promedio
columnas_numericas_spline_promedio = [
    'P_E',
    'P_B',
    'P_S',
    'P_Share',
    'ROCE_sp',
    'EBIT_sp',
    'Total Activos_sp',
    'Deuda a LP_sp',
    'ROA_sp',
    'Beneficio neto_sp',
    'ROI_sp',
    'EV_sp',
    'Cap de mercado_sp',
    'Deuda a CP_sp',
    'Efectivo y equiv_sp',
    'dif_CPI_promedio',
    'dif_FFR_promedio',
    'dif_NFP_promedio'
]

In [11]:
listas_columnas = {
    'lineal_mediana': columnas_numericas_lineal_mediana,
    'spline_mediana': columnas_numericas_spline_mediana,
    'lineal_promedio': columnas_numericas_lineal_promedio,
    'spline_promedio': columnas_numericas_spline_promedio
}

In [12]:
listas_columnas

{'lineal_mediana': ['P_E',
  'P_B',
  'P_S',
  'P_Share',
  'ROCE_l',
  'EBIT_l',
  'Total Activos_l',
  'Deuda a LP_l',
  'ROA_l',
  'Beneficio neto_l',
  'ROI_l',
  'EV_l',
  'Cap de mercado_l',
  'Deuda a CP_l',
  'Efectivo y equiv_l',
  'dif_CPI_mediana',
  'dif_FFR_mediana',
  'dif_NFP_mediana'],
 'spline_mediana': ['P_E',
  'P_B',
  'P_S',
  'P_Share',
  'ROCE_sp',
  'EBIT_sp',
  'Total Activos_sp',
  'Deuda a LP_sp',
  'ROA_sp',
  'Beneficio neto_sp',
  'ROI_sp',
  'EV_sp',
  'Cap de mercado_sp',
  'Deuda a CP_sp',
  'Efectivo y equiv_sp',
  'dif_CPI_mediana',
  'dif_FFR_mediana',
  'dif_NFP_mediana'],
 'lineal_promedio': ['P_E',
  'P_B',
  'P_S',
  'P_Share',
  'ROCE_l',
  'EBIT_l',
  'Total Activos_l',
  'Deuda a LP_l',
  'ROA_l',
  'Beneficio neto_l',
  'ROI_l',
  'EV_l',
  'Cap de mercado_l',
  'Deuda a CP_l',
  'Efectivo y equiv_l',
  'dif_CPI_promedio',
  'dif_FFR_promedio',
  'dif_NFP_promedio'],
 'spline_promedio': ['P_E',
  'P_B',
  'P_S',
  'P_Share',
  'ROCE_sp',
  'E

In [13]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 1: Importaciones y Funciones Auxiliares                  │
# └─────────────────────────────────────────────────────────────────┘
import pandas as pd
import numpy as np

def pct_missing_by_column(df, columns):
    """Devuelve % de NaNs por columna, ordenado descendente."""
    return (df[columns].isna().mean() * 100).sort_values(ascending=False)

def pct_missing_by_group(df, group_col, features):
    """Devuelve Series con el máximo % de NaNs en cualquier feature por grupo."""
    missing = df.groupby(group_col)[features]\
                .apply(lambda g: g.isna().mean()*100)
    return missing.max(axis=1)

def filter_companies_by_target_missing(df, group_col, target_col):
    """Filtra y devuelve df sin empresas que tengan ANY NaN en target_col."""
    ok = ~df.groupby(group_col)[target_col].apply(lambda s: s.isna().any())
    valid_companies = ok[ok].index
    return df[df[group_col].isin(valid_companies)].copy()

def impute_by_group(df, group_col, features, methods=('ffill',), fill_value=0): # <-- Cambiar default
    """Imputa NaNs por grupo usando .ffill() y termina con fillna(fill_value)."""
    df2 = df.sort_values([group_col, 'Fecha']).copy()
    for m in methods:
        df2[features] = df2.groupby(group_col)[features]\
                            .transform(lambda g: getattr(g, m)())
    return df2.fillna(fill_value)

def create_lags(df, group_col, date_col, features, lag=1):
    """Genera columnas de lag para cada feature dentro de cada grupo."""
    df2 = df.sort_values([group_col, date_col]).copy()
    for feat in features:
        df2[f"{feat}_lag{lag}"] = df2.groupby(group_col)[feat].shift(lag)
    return df2


In [14]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 3: Conversión Masiva a Numérico                         │
# └─────────────────────────────────────────────────────────────────┘
# Crear set de features base (sin 'Empresa' ni 'Fecha'), añadir 'P_E' si existe
features_base = {
    col for cols in listas_columnas.values()
            for col in cols if col not in ('Empresa','Fecha')
}
if 'P_E' in df.columns:
    features_base.add('P_E')
numeric_cols = [c for c in sorted(features_base) if c in df.columns]

# Vectorizado
nan_before = df[numeric_cols].isna().sum()
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
nan_after = df[numeric_cols].isna().sum()
print("NaNs añadidos:", (nan_after - nan_before).sum())

NaNs añadidos: 0


In [15]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 4: Filtrar Empresas con NaNs en el Target P_E           │
# └─────────────────────────────────────────────────────────────────┘
df_no_PE_NaN = filter_companies_by_target_missing(df, 'Empresa', 'P_E')
print("Empresas tras filtrar P_E NaN:", df_no_PE_NaN['Empresa'].nunique())

Empresas tras filtrar P_E NaN: 403


In [16]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 5: Excluir Columnas Críticas con Demasiados NaNs        │
# └─────────────────────────────────────────────────────────────────┘
col_pct = pct_missing_by_column(df_no_PE_NaN, numeric_cols)
col_pct

EBIT_sp                29.698098
EBIT_l                 29.698098
EV_sp                  28.662117
EV_l                   28.662117
Deuda a CP_l            6.952026
Deuda a CP_sp           6.952026
ROCE_sp                 3.196857
ROCE_l                  3.196857
ROI_sp                  2.045079
ROI_l                   2.045079
Deuda a LP_sp           1.774194
Deuda a LP_l            1.774194
ROA_l                   1.600496
ROA_sp                  1.600496
P_B                     1.565343
Total Activos_sp        1.428867
Total Activos_l         1.428867
Efectivo y equiv_l      1.418528
Efectivo y equiv_sp     1.418528
Beneficio neto_sp       1.296526
Beneficio neto_l        1.296526
Cap de mercado_sp       1.265509
Cap de mercado_l        1.265509
P_S                     0.051696
P_Share                 0.002068
P_E                     0.000000
dif_CPI_mediana         0.000000
dif_CPI_promedio        0.000000
dif_FFR_mediana         0.000000
dif_FFR_promedio        0.000000
dif_NFP_me

In [17]:
umbral_col = 5.0   # % máximo de NaNs tolerable
cols_to_exclude = col_pct[col_pct > umbral_col].index.tolist()
features_kept = [c for c in numeric_cols if c not in cols_to_exclude]
print(f"Columnas excluidas (> {umbral_col}% NaNs):", cols_to_exclude)

Columnas excluidas (> 5.0% NaNs): ['EBIT_sp', 'EBIT_l', 'EV_sp', 'EV_l', 'Deuda a CP_l', 'Deuda a CP_sp']


In [18]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 6: Excluir Empresas con Demasiados NaNs en Features     │
# └─────────────────────────────────────────────────────────────────┘
umbral_grp = 10   # % máximo de NaNs por empresa
grp_pct = pct_missing_by_group(df_no_PE_NaN, 'Empresa', features_kept)
empresas_final = grp_pct[grp_pct <= umbral_grp].index.tolist()
df_final = df_no_PE_NaN[df_no_PE_NaN['Empresa'].isin(empresas_final)].copy()
print("Empresas finales (<= {umbral_grp}% NaNs):", len(empresas_final))


Empresas finales (<= {umbral_grp}% NaNs): 381


In [19]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 7: Imputar NaNs Restantes                                │
# └─────────────────────────────────────────────────────────────────┘
df_imputed = impute_by_group(df_final, 'Empresa', features_kept)
print("NaNs totales después de imputar:", df_imputed[features_kept].isna().sum().sum())


NaNs totales después de imputar: 0


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │ BLOQUE 8: Crear DataFrames con Lag (Incluyendo P_E_lag1 como feature) │
# └─────────────────────────────────────────────────────────────────┘
dfs_shifted = {}
LAG_VALUE_EXPERIMENT = 1 # Define el lag a usar (ej. 1 para _lag1)

# Asunciones:
# 1. df_imputed: DataFrame con datos limpios, incluyendo 'Empresa', 'Fecha', 'P_E' y las columnas base.
# 2. listas_columnas: Diccionario {metodo: [lista_de_columnas_base_para_ese_metodo]}
# 3. features_kept: Lista de columnas que pasaron filtros de NaNs (debe incluir 'P_E' si quieres su lag).
#    Si 'P_E' no está en features_kept porque se trató solo como target, necesitaremos añadirlo
#    específicamente para la creación de su lag.

# Asegurarse de que 'P_E' esté considerado para laggear si queremos P_E_lag1
# Esto es importante si 'features_kept' no incluye 'P_E' porque se manejó solo como target antes.
features_kept_for_lags = features_kept.copy() # Trabajar con una copia
if 'P_E' not in features_kept_for_lags and 'P_E' in df_imputed.columns:
    print("  Nota Bloque 8: Añadiendo 'P_E' a la lista de features a considerar para lags, para poder generar P_E_lagN.")
    features_kept_for_lags.append('P_E')
    features_kept_for_lags = sorted(list(set(features_kept_for_lags)))


for metodo, cols_originales_del_metodo in listas_columnas.items():
    print(f"\nProcesando lags para Método: {metodo}")
    # 1. Determinar las features base para este método que SÍ existen y pasaron filtros
    #    Estas son las features (EXCLUYENDO IDs y el target P/E original) para las que se crearán lags.
    #    PERO, si queremos P_E_lag1, P_E debe estar entre las features para las que se crea un lag.
    
    features_base_para_este_metodo = [
        col for col in cols_originales_del_metodo # Empezar con las definidas para el método
        if col in features_kept_for_lags  # Solo las que pasaron el filtro de NaNs global (y ahora incluye P_E)
           and col in df_imputed.columns    # Y existen en df_imputed
           and col not in ('Empresa', 'Fecha') # Excluir IDs
           # NO excluimos 'P_E' aquí porque queremos generar P_E_lagN
    ]

    if not features_base_para_este_metodo:
        print(f"  Método '{metodo}': No hay features base válidas (de 'features_kept_for_lags') para crear lags. Se omite.")
        continue
    
    print(f"  Features base para {metodo} que se usarán para generar lags ({len(features_base_para_este_metodo)}): {features_base_para_este_metodo[:5]}...")

    # 2. Preparar el DataFrame base para la función create_lags
    #    Necesita 'Empresa', 'Fecha', y TODAS las 'features_base_para_este_metodo' (que ahora incluye 'P_E')
    columnas_para_df_con_lags = ['Empresa', 'Fecha']
    # Añadir las features base, asegurando que 'P_E' (target original) también esté si no está ya en features_base_para_este_metodo
    # (aunque debería estarlo si se añadió a features_kept_for_lags)
    columnas_para_df_con_lags.extend(f for f in features_base_para_este_metodo if f not in columnas_para_df_con_lags)
    if 'P_E' not in columnas_para_df_con_lags and 'P_E' in df_imputed.columns: # Asegurar que P_E (target) esté para el join final
        columnas_para_df_con_lags.append('P_E')

    columnas_para_df_con_lags = [col for col in columnas_para_df_con_lags if col in df_imputed.columns] # Solo existentes
    
    if not all(essential_col in columnas_para_df_con_lags for essential_col in ['Empresa', 'Fecha']):
        print(f"  Método '{metodo}': Faltan 'Empresa' o 'Fecha'. Se omite.")
        continue
        
    df_para_crear_lags = df_imputed[columnas_para_df_con_lags].copy()

    # 3. Generar los lags para TODAS las features_base_para_este_metodo (incluyendo P_E)
    #    La función create_lags añadirá columnas como 'FeatureX_lag1', 'P_E_lag1'
    df_con_lags_generados = create_lags(df_para_crear_lags, 'Empresa', 'Fecha', features_base_para_este_metodo, lag=LAG_VALUE_EXPERIMENT)

    # 4. Definir las columnas de lag que se usarán como predictores finales
    #    Esto incluye P_E_lagN y los lags de las otras features.
    predictor_cols_finales_con_lag = [f"{feat}_lag{LAG_VALUE_EXPERIMENT}" for feat in features_base_para_este_metodo]
    
    # Asegurar que estas columnas de lag realmente se generaron
    predictor_cols_finales_existentes = [col for col in predictor_cols_finales_con_lag if col in df_con_lags_generados.columns]

    if not predictor_cols_finales_existentes:
         print(f"  Método '{metodo}': No se generaron columnas de lag válidas. Se omite.")
         continue
    
    # 5. Eliminar filas donde CUALQUIERA de los predictores de lag finales sea NaN
    rows_before_dropna = len(df_con_lags_generados)
    df_con_lags_generados.dropna(subset=predictor_cols_finales_existentes, inplace=True)
    rows_after_dropna = len(df_con_lags_generados)
    print(f"  {metodo}: Se eliminaron {rows_before_dropna - rows_after_dropna} filas por NaNs en lags.")


    if df_con_lags_generados.empty:
        print(f"  Método '{metodo}': El DataFrame quedó vacío después de crear lags y dropna.")
    else:
        # El DataFrame final para este método contendrá:
        # 'Empresa', 'Fecha', 'P_E' (target actual), y todas las 'predictor_cols_finales_existentes'
        # (que incluye P_E_lagN y los lags de otras features)
        
        # Seleccionar solo las columnas necesarias para el siguiente bloque (experimentos)
        columnas_a_mantener_en_dfs_shifted = ['Empresa', 'Fecha']
        if 'P_E' in df_con_lags_generados.columns: # Target
            columnas_a_mantener_en_dfs_shifted.append('P_E')
        columnas_a_mantener_en_dfs_shifted.extend(predictor_cols_finales_existentes) # Todas las features _lagN
        
        # Eliminar duplicados en la lista de columnas por si acaso
        columnas_a_mantener_en_dfs_shifted = sorted(list(set(col for col in columnas_a_mantener_en_dfs_shifted if col in df_con_lags_generados.columns)))

        dfs_shifted[metodo] = df_con_lags_generados[columnas_a_mantener_en_dfs_shifted].copy()
        
        print(f"  Método '{metodo}' (dfs_shifted): {dfs_shifted[metodo].shape[0]} filas, {dfs_shifted[metodo].shape[1]} columnas.")
        print(f"    Predictores finales (lags): {len(predictor_cols_finales_existentes)} {predictor_cols_finales_existentes[:5]}...")
        if f'P_E_lag{LAG_VALUE_EXPERIMENT}' in predictor_cols_finales_existentes:
            print(f"    ¡Confirmado! 'P_E_lag{LAG_VALUE_EXPERIMENT}' está incluido como predictor.")

In [23]:
# Celda: BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON LAGS Y GUARDAR PAQUETE COMPLETO)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib

# --- Modelos ---
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Métricas y Preprocesamiento ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupKFold # BaseCrossValidator no se usa directamente
from sklearn.pipeline import Pipeline

# Ignorar warnings comunes (opcional)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================
# CLASE AUXILIAR DropColumns (Definir una vez)
# =====================================
class DropColumns(BaseEstimator, TransformerMixin):
    """Transformer para eliminar columnas especificadas en un Pipeline."""
    def __init__(self, columns=None):
        self.columns = columns if columns is not None else []
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            return X
        X_transformed = X.copy()
        return X_transformed.drop(columns=self.columns, errors='ignore')

# =====================================
# FUNCIONES AUXILIARES
# =====================================
def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_test_split_by_empresa(df_model_input, n_empresas_prueba):
    df_model = df_model_input.copy()
    if 'Empresa' not in df_model.columns: raise KeyError("'Empresa' necesaria")
    if 'P_E' not in df_model.columns: raise KeyError("'P_E' (target) necesaria")

    empresas_unicas = df_model['Empresa'].unique()
    n_total_empresas = len(empresas_unicas)

    if n_total_empresas == 0: return None, None, None, None
    if n_total_empresas == 1:
        X = df_model.drop(columns=['P_E']); y = df_model['P_E']
        return X.copy(), y.copy(), pd.DataFrame(columns=X.columns), pd.Series(dtype=y.dtype)

    # Asegurar que n_empresas_prueba sea válido (entre 0 y n_total_empresas - 1)
    n_empresas_prueba_valido = max(0, min(n_empresas_prueba, n_total_empresas - 1 if n_total_empresas > 0 else 0))
    if n_empresas_prueba_valido != n_empresas_prueba:
        print(f"  Advertencia train_test_split: n_empresas_prueba ajustado de {n_empresas_prueba} a {n_empresas_prueba_valido}")
    n_empresas_prueba = n_empresas_prueba_valido


    if n_empresas_prueba == 0:
        empresas_entrenamiento = empresas_unicas; empresas_prueba = np.array([])
    else:
        empresas_entrenamiento, empresas_prueba = train_test_split(
            empresas_unicas, test_size=n_empresas_prueba, random_state=42, shuffle=True)

    mask_entrenamiento = df_model['Empresa'].isin(empresas_entrenamiento)
    X = df_model.drop(columns=['P_E']); y = df_model['P_E']
    X_train, y_train = X[mask_entrenamiento].copy(), y[mask_entrenamiento].copy()

    if n_empresas_prueba > 0 and len(empresas_prueba) > 0:
        mask_prueba = df_model['Empresa'].isin(empresas_prueba)
        X_test, y_test = X[mask_prueba].copy(), y[mask_prueba].copy()
        # Si X_test es vacío pero se esperaban datos de prueba (mask_prueba no vacía), inicializarlo vacío con columnas correctas
        if X_test.empty and not mask_prueba.empty() and not X_train.empty : 
             X_test = pd.DataFrame(columns=X_train.columns)
             y_test = pd.Series(dtype=y_train.dtype)

    else: # No hay empresas de prueba o X_train está vacío
        X_test_cols = X_train.columns if not X_train.empty else (X.columns if not X.empty else [])
        y_test_dtype = y_train.dtype if not y_train.empty else (y.dtype if not y.empty else float)
        X_test, y_test = pd.DataFrame(columns=X_test_cols), pd.Series(dtype=y_test_dtype)


    if X_train.empty and n_total_empresas > 0 : # Si X_train está vacío pero había datos, es un error
        print("ERROR train_test_split: X_train está vacío después del split.")
        return None, None, None, None
        
    return X_train, y_train, X_test, y_test


def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    
    mae = mean_absolute_error(y_test_np, y_pred_np)
    rmse_val = rmse(y_test_np, y_pred_np) # rmse ya maneja NaNs/Infs
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

def sort_by_timestep(X_df, y_series):
    X, y = X_df.copy(), y_series.copy()
    if 'time_step' not in X.columns: return X, y
    if not isinstance(y, pd.Series) or not X.index.equals(y.index): # Asegurar que los índices coincidan
        if len(y) == len(X): y = pd.Series(np.asarray(y), index=X.index, name=getattr(y, 'name', 'target'))
        else: print("Error sort_by_timestep: y no alineable con X."); return X, y
    X_sorted = X.sort_values('time_step'); y_sorted = y.loc[X_sorted.index]
    return X_sorted, y_sorted

# ================================================================================
# BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON CARACTERÍSTICAS REZAGADAS '_lag1')
# ================================================================================

# ASUNCIONES ANTES DE CORRER ESTE BLOQUE:
# 1. `dfs_shifted`: Diccionario. Key=`metodo`. Value (`df_lag1_original_con_todo`) es un DataFrame que contiene:
#      - IDs: 'Empresa', 'Fecha'.
#      - Target: 'P_E'.
#      - COLUMNAS BASE ORIGINALES del método.
#      - COLUMNAS REZAGADAS `_lag1` (el número de estas depende de filtros previos como `features_kept` en BLOQUE 8).
#    El número de columnas `_lag1` en `dfs_shifted[metodo]` determinará las features para el modelo de ESE `metodo`.

results = []
start_time_total = time.time()
use_log_transform = True
test_size_ratio = 0.20

for metodo, df_lag1_original_con_todo in dfs_shifted.items():
    start_time_config = time.time()
    print("\n" + "="*80)
    print(f"Procesando Método: {metodo} {'(CON Log Transform)' if use_log_transform else '(SIN Log Transform)'} (SOLO LAGS COMO PREDICTORES)")
    print("="*80 + "\n")

    if df_lag1_original_con_todo.empty:
        print(f"  ⇨ ADVERTENCIA: df_lag1_original_con_todo para el método {metodo} está vacío. Omitiendo.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': [], 'n_features_modelo': 0, 'Error': 'Input DataFrame for method is empty'})
        continue

    df_temp_for_method = df_lag1_original_con_todo.copy()

    if 'time_step' not in df_temp_for_method.columns:
        df_temp_for_method.sort_values(['Empresa', 'Fecha'], inplace=True)
        df_temp_for_method['time_step'] = df_temp_for_method.groupby('Empresa').cumcount()

    predictor_cols_solo_lag1 = sorted([c for c in df_temp_for_method.columns if c.endswith('_lag1')])
    
    if not predictor_cols_solo_lag1:
        print(f"  ⇨ ADVERTENCIA: No se encontraron columnas _lag1 en dfs_shifted['{metodo}']. Omitiendo método {metodo}.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': [], 'n_features_modelo': 0, 'Error': f'No lag features found in dfs_shifted for {metodo}'})
        continue
    
    columnas_base_que_generaron_estos_lags = sorted(list(set(lag_col.replace('_lag1', '') for lag_col in predictor_cols_solo_lag1)))
    print(f"  ⇨ Método {metodo}: Se usarán {len(predictor_cols_solo_lag1)} predictores (_lag1).")
    print(f"    Estos lags fueron generados a partir de {len(columnas_base_que_generaron_estos_lags)} columnas base: {columnas_base_que_generaron_estos_lags[:5]}...")

    cols_for_df_model = ['Empresa', 'Fecha', 'time_step', 'P_E'] + predictor_cols_solo_lag1
    final_cols_for_df_model = sorted(list(set(c for c in cols_for_df_model if c in df_temp_for_method.columns))) # Unicas y existentes
    
    essential_check = ['Empresa', 'Fecha', 'time_step', 'P_E']
    if not all(ec in final_cols_for_df_model for ec in essential_check):
        missing_ess_cols = [ec for ec in essential_check if ec not in final_cols_for_df_model]
        print(f"  ⇨ ERROR: Faltan columnas esenciales ({missing_ess_cols}) para construir df_model para {metodo}. Omitiendo.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags, 'n_features_modelo': len(predictor_cols_solo_lag1), 'Error': f'Missing essential cols for df_model: {missing_ess_cols}'})
        continue
        
    df_model = df_temp_for_method[final_cols_for_df_model].copy()

    n_total_empresas_metodo = df_model['Empresa'].nunique()
    n_empresas_test_metodo = 0
    if n_total_empresas_metodo > 1:
        n_empresas_test_metodo = max(1, int(round(n_total_empresas_metodo * test_size_ratio)))
        if n_total_empresas_metodo - n_empresas_test_metodo < 1 : n_empresas_test_metodo = n_total_empresas_metodo - 1
    
    X_train, y_train, X_test, y_test = train_test_split_by_empresa(df_model, n_empresas_test_metodo)
    
    if X_train is None or X_train.empty:
        print(f"  Split train/test fallido o X_train vacío para {metodo}. Omitiendo método.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags, 'n_features_modelo': len(predictor_cols_solo_lag1), 'Error': 'Train/Test split failed or X_train empty'})
        continue
    
    print(f"  ⇨ Shapes post-split: X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape if X_test is not None else 'None'}, y_test: {y_test.shape if y_test is not None else 'None'}")

    y_train_transformed = y_train.copy()
    if use_log_transform:
        y_numeric = pd.to_numeric(y_train, errors='coerce')
        median_y = y_numeric.median() if not y_numeric.isnull().all() else 0.0
        y_numeric = y_numeric.fillna(median_y) # Rellenar NaNs antes de chequear negativos
        y_numeric.loc[y_numeric < 0] = 0 # Asignar 0 a negativos para log1p
        y_train_transformed = np.log1p(y_numeric)
        median_yt = y_train_transformed.median() if not y_train_transformed.isnull().all() else 0.0
        y_train_transformed = y_train_transformed.fillna(median_yt)


    X_train_s, y_train_s = sort_by_timestep(X_train, y_train_transformed)
    X_test_s, y_test_s_original = pd.DataFrame(columns=X_train.columns), pd.Series(dtype=y_train.dtype) # Inicializar
    if X_test is not None and not X_test.empty:
        X_test_s, y_test_s_original = sort_by_timestep(X_test, y_test)

    param_grid_rf   = {'model__max_depth':[10, 20, None],'model__min_samples_split':[5, 10],'model__n_estimators':[100, 200]}
    param_grid_xgb  = {'model__n_estimators':[100, 200],'model__max_depth':[3, 5, 7],'model__learning_rate':[0.1, 0.05]}
    param_grid_lgbm = {'model__n_estimators':[100, 200],'model__learning_rate':[0.1, 0.05],'model__max_depth':[3, 5, 7],'model__num_leaves':[15, 31]}
    param_grid_cb   = {'model__iterations':[100, 200],'model__learning_rate':[0.1, 0.05],'model__depth':[3, 5, 7],'model__l2_leaf_reg':[1, 3]}

    scenarios = {
        'RF_Simple':  {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False},
        'RF_HP':      {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_rf},
        'XGB_Simple': {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False},
        'XGB_HP':     {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_xgb},
        'LGBM_Simple':{'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': False, 'vt': False},
        'LGBM_HP':    {'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': True,  'vt': False, 'grid': param_grid_lgbm},
        'CB_Simple':  {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': False, 'vt': False},
        'CB_HP':      {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': True,  'vt': False, 'grid': param_grid_cb},
    }

    for escenario_nombre, escenario_cfg in scenarios.items():
        tiempo_esc_inicio = time.time()
        print(f"\n  --- Escenario: {escenario_nombre} para Método: {metodo} ---")
        current_base_model, perform_hp_search, use_temporal_cv = escenario_cfg['model'], escenario_cfg['hp'], escenario_cfg['vt']
        X_fit_data, y_fit_data = (X_train_s, y_train_s) if use_temporal_cv else (X_train, y_train_transformed)
        X_eval_data, y_eval_data_original = (X_test_s, y_test_s_original) if use_temporal_cv else (X_test, y_test)

        pipeline_obj = Pipeline([('drop_ids', DropColumns(columns=['Empresa', 'Fecha', 'time_step'])), ('model', current_base_model)])
        final_fitted_model, hp_best_params, error_msg_escenario = None, None, None

        try:
            if perform_hp_search:
                cv_strategy, cv_fit_params = None, {}
                n_groups_cv = X_fit_data['Empresa'].nunique() if 'Empresa' in X_fit_data.columns else 0

                can_do_cv = True
                if use_temporal_cv and n_groups_cv >= 2:
                    X_cv_input = X_fit_data.sort_values(['Empresa', 'time_step']); y_cv_input = y_fit_data.loc[X_cv_input.index]
                    groups_cv = X_cv_input['Empresa']
                    n_splits = min(4, n_groups_cv); n_splits = max(2, n_splits) # GroupKFold min 2
                    if len(X_cv_input) < n_splits: can_do_cv = False; print(f"    Pocas muestras ({len(X_cv_input)}) para {n_splits} splits en GroupKFold.")
                    else: cv_strategy = GroupKFold(n_splits=n_splits); cv_fit_params = {'groups': groups_cv}; print(f"    Usando GroupKFold CV con {n_splits} splits.")
                elif len(X_fit_data) >= 2 : # KFold estándar
                    kfold_splits = min(3, len(X_fit_data)); kfold_splits = max(2, kfold_splits) # KFold min 2
                    if len(X_fit_data) < kfold_splits : can_do_cv = False; print(f"    Pocas muestras ({len(X_fit_data)}) para {kfold_splits} splits en KFold.")
                    else: cv_strategy = KFold(n_splits=kfold_splits, shuffle=True, random_state=42); print(f"    Usando KFold CV con {kfold_splits} splits.")
                else: can_do_cv = False; print(f"    No hay suficientes muestras ({len(X_fit_data)}) para CV. Omitiendo HP search.")
                
                if can_do_cv:
                    gs = GridSearchCV(pipeline_obj, escenario_cfg['grid'], scoring=rmse_scorer, cv=cv_strategy, n_jobs=-1, refit=True, error_score='raise', verbose=0)
                    gs.fit(X_fit_data, y_fit_data, **cv_fit_params)
                    final_fitted_model, hp_best_params = gs.best_estimator_, gs.best_params_
                    print(f"    Mejores parámetros: {hp_best_params}")
                else: perform_hp_search = False # Forzar no HP search y fit simple
            
            if not perform_hp_search or not final_fitted_model: # Si HP search se omitió o falló, fit simple
                pipeline_obj.fit(X_fit_data, y_fit_data)
                final_fitted_model = pipeline_obj
        except Exception as e: error_msg_escenario = str(e); print(f"    ERROR entrenamiento/HP search {escenario_nombre}: {e}")

        eval_metrics, model_n_features = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}, "N/A"
        if final_fitted_model:
            try:
                model_estimator_step = final_fitted_model.steps[-1][1]
                model_n_features = getattr(model_estimator_step, 'n_features_in_', 'N/A')
                print(f"    >> Estimador final '{type(model_estimator_step).__name__}' vio {model_n_features} features.")
                if isinstance(model_n_features, int) and model_n_features != len(predictor_cols_solo_lag1):
                     print(f"       ¡ADVERTENCIA! n_features ({model_n_features}) no coincide con num. de lags ({len(predictor_cols_solo_lag1)}).")

                if X_eval_data is not None and not X_eval_data.empty and y_eval_data_original is not None and not y_eval_data_original.empty:
                    pred_transformed = final_fitted_model.predict(X_eval_data)
                    pred_original_scale = np.expm1(pred_transformed) if use_log_transform else pred_transformed
                    if use_log_transform: pred_original_scale[~np.isfinite(pred_original_scale)] = np.nan
                    eval_metrics = evaluate_model(y_eval_data_original, pred_original_scale)
                    print(f"    Métricas Test {escenario_nombre}: RMSE={eval_metrics['RMSE']:.4f}, MAE={eval_metrics['MAE']:.4f}, R2={eval_metrics['R2']:.4f}")
                else: print("    No hay datos de X_eval/y_eval para evaluar.")
            except Exception as e_eval: error_msg_escenario = (error_msg_escenario + f" | EvalError: {str(e_eval)}") if error_msg_escenario else f"EvalError: {str(e_eval)}"; print(f"    ERROR evaluación {escenario_nombre}: {e_eval}")
        
        results.append({
            'Modelo': escenario_nombre.split('_')[0], 'Validacion_Temp': use_temporal_cv,
            'Busqueda_HP': perform_hp_search and escenario_cfg['hp'], 'Config_Key': metodo, 'Metodo': metodo,
            'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': eval_metrics['RMSE'],
            'MAE': eval_metrics['MAE'], 'R2': eval_metrics['R2'], 'Best_Params': hp_best_params,
            'pipeline': final_fitted_model, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags,
            'n_features_modelo': model_n_features, 'Error': error_msg_escenario})
        print(f"    Tiempo escenario {escenario_nombre}: {time.time() - tiempo_esc_inicio:.2f}s")
    print(f"\nTiempo total Método {metodo}: {time.time() - start_time_config:.2f}s")
print(f"\n\nTiempo total ejecución experimentos: {(time.time() - start_time_total)/60:.2f} minutos")

# ================================================================================
# CREAR TABLA DE RESULTADOS Y GUARDAR MEJOR MODELO (PAQUETE COMPLETO)
# ================================================================================
if results:
    df_results_all_models = pd.DataFrame(results)
    print("\n" + "="*80 + "\n--- Resultados Finales (Modelos Entrenados SOLO CON LAGS) ---\n" + f"Total resultados: {len(df_results_all_models)}\n")
    cols_display_final = ['Metodo', 'Modelo', 'Busqueda_HP', 'RMSE', 'MAE', 'R2', 'n_features_modelo', 'columnas_base_usadas', 'Best_Params', 'Error']
    cols_to_show_final = [col for col in cols_display_final if col in df_results_all_models.columns]
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:.4f}'.format):
        print(df_results_all_models.sort_values(by='RMSE', ascending=True, na_position='last')[cols_to_show_final].to_string(index=False))

    valid_results_for_best = df_results_all_models[df_results_all_models['RMSE'].notna() & df_results_all_models['pipeline'].notna() & df_results_all_models['Error'].isna()].copy()
    if not valid_results_for_best.empty:
        best_model_idx = valid_results_for_best['RMSE'].idxmin()
        best_model_full_info = valid_results_for_best.loc[best_model_idx]
        
        model_package_to_save = {
            'pipeline': best_model_full_info['pipeline'],
            'metodo_features': best_model_full_info['Metodo'], # Clave corregida
            'log_transform_target': best_model_full_info['Log_Transform'],
            'columnas_base_para_lags': best_model_full_info['columnas_base_usadas'],
            'n_features_in_estimator': best_model_full_info['n_features_modelo']}
        
        filename_best_model_package = f"best_model_package_{best_model_full_info['Metodo']}.joblib"
        joblib.dump(model_package_to_save, filename_best_model_package)
        
        print("\n" + "="*80 + "\nPAQUETE DEL MEJOR MODELO (SOLO LAGS) SELECCIONADO Y GUARDADO:\n" +
              f"  Archivo: {filename_best_model_package}\n" +
              f"  Método de Features Original: {best_model_full_info['Metodo']}\n" +
              f"  Columnas Base (de los lags usados): {len(best_model_full_info['columnas_base_usadas'])} {str(best_model_full_info['columnas_base_usadas'][:10]) + ('...' if len(best_model_full_info['columnas_base_usadas']) > 10 else '')}\n" +
              f"  Num. Features para Estimador (lags): {best_model_full_info['n_features_modelo']}\n" +
              f"  Transformación Log Target: {best_model_full_info['Log_Transform']}\n" +
              f"  RMSE en Test: {best_model_full_info['RMSE']:.4f}\n" +
              f"  MAE en Test: {best_model_full_info['MAE']:.4f}\n" +
              f"  R2 en Test: {best_model_full_info['R2']:.4f}\n" +
              f"  Hiperparámetros: {best_model_full_info['Best_Params']}\n" +
              f"  Pipeline: {best_model_full_info['pipeline']}\n" + "="*80)
    else: print("\nNo se encontraron resultados válidos para seleccionar y guardar el mejor modelo.")
else: print("\nNo se generaron resultados en los experimentos.")


Procesando Método: lineal_mediana (CON Log Transform) (SOLO LAGS COMO PREDICTORES)

  ⇨ Método lineal_mediana: Se usarán 14 predictores (_lag1).
    Estos lags fueron generados a partir de 14 columnas base: ['Beneficio neto_l', 'Cap de mercado_l', 'Deuda a LP_l', 'Efectivo y equiv_l', 'P_B']...
  ⇨ Shapes post-split: X_train: (36295, 17), y_train: (36295,), X_test: (9044, 17), y_test: (9044,)

  --- Escenario: RF_Simple para Método: lineal_mediana ---
    >> Estimador final 'RandomForestRegressor' vio 14 features.
    Métricas Test RF_Simple: RMSE=26.9506, MAE=7.4356, R2=0.2771
    Tiempo escenario RF_Simple: 11.68s

  --- Escenario: RF_HP para Método: lineal_mediana ---
    Usando KFold CV con 3 splits.
    Mejores parámetros: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 100}
    >> Estimador final 'RandomForestRegressor' vio 14 features.
    Métricas Test RF_HP: RMSE=26.9566, MAE=7.4463, R2=0.2768
    Tiempo escenario RF_HP: 280.90s

  --- Escenar

In [24]:
print("\nTabla de Resultados:")
df_results_all_models


Tabla de Resultados:


,Modelo,Validacion_Temp,Busqueda_HP,Config_Key,Metodo,Lag_Config,Log_Transform,RMSE,MAE,R2,Best_Params,pipeline,columnas_base_usadas,n_features_modelo,Error
0,RF,False,False,lineal_mediana,lineal_mediana,lag1_solo_pred,True,26.950645,7.435600,0.277147,None,"(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
1,RF,False,True,lineal_mediana,lineal_mediana,lag1_solo_pred,True,26.956563,7.446329,0.276830,"{'model__max_depth': None, 'model__min_samples...","(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
2,XGB,False,False,lineal_mediana,lineal_mediana,lag1_solo_pred,True,27.227543,7.263090,0.262217,None,"(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
3,XGB,False,True,lineal_mediana,lineal_mediana,lag1_solo_pred,True,26.612254,6.973780,0.295186,"{'model__learning_rate': 0.1, 'model__max_dept...","(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
4,LGBM,False,False,lineal_mediana,lineal_mediana,lag1_solo_pred,True,26.753230,6.981379,0.287698,None,"(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
5,LGBM,False,True,lineal_mediana,lineal_mediana,lag1_solo_pred,True,26.499791,6.874856,0.301130,"{'model__learning_rate': 0.1, 'model__max_dept...","(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
6,CB,False,False,lineal_mediana,lineal_mediana,lag1_solo_pred,True,26.865057,6.771961,0.281731,None,"(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
7,CB,False,True,lineal_mediana,lineal_mediana,lag1_solo_pred,True,26.653554,6.873709,0.292996,"{'model__depth': 7, 'model__iterations': 200, ...","(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
8,RF,False,False,spline_mediana,spline_mediana,lag1_solo_pred,True,27.183906,7.452823,0.264580,None,"(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_sp, Cap de mercado_sp, Deuda a...",14,None
9,RF,False,True,spline_mediana,spline_mediana,lag1_solo_pred,True,27.092927,7.451175,0.269495,"{'model__max_depth': None, 'model__min_samples...","(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_sp, Cap de mercado_sp, Deuda a...",14,None


In [30]:
# Elección entre modelos

def select_candidate_models(df, n=3):
    """
    Selecciona los n modelos candidatos basándose únicamente en el RMSE (de menor a mayor).
    """
    return df.sort_values('RMSE').head(n)

In [26]:
# Candidatos a mejor modelo
top_candidates = select_candidate_models(df_results_all_models, n=3)
print("Top candidatos según RMSE")
top_candidates

Top candidatos según RMSE


,Modelo,Validacion_Temp,Busqueda_HP,Config_Key,Metodo,Lag_Config,Log_Transform,RMSE,MAE,R2,Best_Params,pipeline,columnas_base_usadas,n_features_modelo,Error
21,LGBM,False,True,lineal_promedio,lineal_promedio,lag1_solo_pred,True,26.424985,6.913758,0.305070,"{'model__learning_rate': 0.1, 'model__max_dept...","(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
5,LGBM,False,True,lineal_mediana,lineal_mediana,lag1_solo_pred,True,26.499791,6.874856,0.301130,"{'model__learning_rate': 0.1, 'model__max_dept...","(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None
19,XGB,False,True,lineal_promedio,lineal_promedio,lag1_solo_pred,True,26.521698,6.963417,0.299974,"{'model__learning_rate': 0.1, 'model__max_dept...","(DropColumns(columns=['Empresa', 'Fecha', 'tim...","[Beneficio neto_l, Cap de mercado_l, Deuda a L...",14,None


## Dlocal

In [23]:
# Cargar los datos desde el archivo Excel
file_path = 'C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Dataset/dataset_dlocal_casi sin vacías.xlsx'
df_dlocal = pd.read_excel(file_path, sheet_name='dataset')

In [24]:
df_dlocal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Empresa                         28 non-null     object 
 1   Fecha                           28 non-null     int64  
 2   P_E                             28 non-null     float64
 3   P_B                             28 non-null     float64
 4   P_S                             28 non-null     float64
 5   P_Share                         28 non-null     float64
 6   ROCE_sp                         28 non-null     float64
 7   ROCE_l                          28 non-null     float64
 8   EBIT_sp                         28 non-null     float64
 9   EBIT_l                          28 non-null     float64
 10  Total Activos_sp                28 non-null     float64
 11  Total Activos_l                 28 non-null     float64
 12  Deuda a LP_sp                   28 non

In [25]:
# Quitar "k" de las columnas "Non farm payrolls_Exp_mediana" y "Non farm payrolls_Exp_promedio"
df_dlocal['Non farm payrolls_Exp_mediana'] = df_dlocal['Non farm payrolls_Exp_mediana'].astype(str).str.replace('k', '')
df_dlocal['Non farm payrolls_Exp_promedio'] = df_dlocal['Non farm payrolls_Exp_promedio'].astype(str).str.replace('k', '')

# Convertir las columnas a valores numéricos
df_dlocal['Non farm payrolls_Exp_mediana'] = pd.to_numeric(df_dlocal['Non farm payrolls_Exp_mediana'], errors='coerce')
df_dlocal['Non farm payrolls_Exp_promedio'] = pd.to_numeric(df_dlocal['Non farm payrolls_Exp_promedio'], errors='coerce')

df_dlocal['P_B'] = pd.to_numeric(df_dlocal['P_B'], errors='coerce')
#df_dlocal['P_TB'] = pd.to_numeric(df_dlocal['P_TB'], errors='coerce')
df_dlocal['P_S'] = pd.to_numeric(df_dlocal['P_S'], errors='coerce')
#df_dlocal['P_FCF'] = pd.to_numeric(df_dlocal['P_FCF'], errors='coerce')
df_dlocal['P_Share'] = pd.to_numeric(df_dlocal['P_Share'], errors='coerce')

In [26]:
# Convertir la columna de fecha a formato datetime
df_dlocal['Fecha'] = pd.to_datetime(df_dlocal['Fecha'], format='%Y%m%d')

# Extraer características temporales relevantes
df_dlocal['mes'] = df_dlocal['Fecha'].dt.month
df_dlocal['año'] = df_dlocal['Fecha'].dt.year
df_dlocal['periodo'] = df_dlocal['Fecha'].dt.to_period('M')

In [27]:
# Convertir CPI y Fed Funds Rate a decimales
df_dlocal['CPI'] = df_dlocal['CPI'] / 100
df_dlocal['Fed Funds Rate'] = df_dlocal['Fed Funds Rate'] / 100

In [28]:
# Calcular diferencias en puntos básicos
df_dlocal['dif_CPI_mediana'] = (df_dlocal['CPI'] - df_dlocal['CPI_Exp_mediana']) * 10000
df_dlocal['dif_CPI_promedio'] = (df_dlocal['CPI'] - df_dlocal['CPI_Exp_promedio']) * 10000

df_dlocal['dif_FFR_mediana'] = (df_dlocal['Fed Funds Rate'] - df_dlocal['Fed Funds Rate_Exp_mediana']) * 10000
df_dlocal['dif_FFR_promedio'] = (df_dlocal['Fed Funds Rate'] - df_dlocal['Fed Funds Rate_Exp_promedio']) * 10000

# Calcular diferencia
df_dlocal['dif_NFP_mediana'] = (df_dlocal['Non farm payrolls'] - df_dlocal['Non farm payrolls_Exp_mediana'])
df_dlocal['dif_NFP_promedio'] = (df_dlocal['Non farm payrolls'] - df_dlocal['Non farm payrolls_Exp_promedio'])

#### Sin transformación logarítmica

In [ ]:
# Celda: BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON LAGS Y GUARDAR PAQUETE COMPLETO)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib

# --- Modelos ---
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Métricas y Preprocesamiento ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupKFold # BaseCrossValidator no se usa directamente
from sklearn.pipeline import Pipeline

# Ignorar warnings comunes (opcional)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================
# CLASE AUXILIAR DropColumns (Definir una vez)
# =====================================
class DropColumns(BaseEstimator, TransformerMixin):
    """Transformer para eliminar columnas especificadas en un Pipeline."""
    def __init__(self, columns=None):
        self.columns = columns if columns is not None else []
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            return X
        X_transformed = X.copy()
        return X_transformed.drop(columns=self.columns, errors='ignore')

# =====================================
# FUNCIONES AUXILIARES
# =====================================
def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_test_split_by_empresa(df_model_input, n_empresas_prueba):
    df_model = df_model_input.copy()
    if 'Empresa' not in df_model.columns: raise KeyError("'Empresa' necesaria")
    if 'P_E' not in df_model.columns: raise KeyError("'P_E' (target) necesaria")

    empresas_unicas = df_model['Empresa'].unique()
    n_total_empresas = len(empresas_unicas)

    if n_total_empresas == 0: return None, None, None, None
    if n_total_empresas == 1:
        X = df_model.drop(columns=['P_E']); y = df_model['P_E']
        return X.copy(), y.copy(), pd.DataFrame(columns=X.columns), pd.Series(dtype=y.dtype)

    # Asegurar que n_empresas_prueba sea válido (entre 0 y n_total_empresas - 1)
    n_empresas_prueba_valido = max(0, min(n_empresas_prueba, n_total_empresas - 1 if n_total_empresas > 0 else 0))
    if n_empresas_prueba_valido != n_empresas_prueba:
        print(f"  Advertencia train_test_split: n_empresas_prueba ajustado de {n_empresas_prueba} a {n_empresas_prueba_valido}")
    n_empresas_prueba = n_empresas_prueba_valido


    if n_empresas_prueba == 0:
        empresas_entrenamiento = empresas_unicas; empresas_prueba = np.array([])
    else:
        empresas_entrenamiento, empresas_prueba = train_test_split(
            empresas_unicas, test_size=n_empresas_prueba, random_state=42, shuffle=True)

    mask_entrenamiento = df_model['Empresa'].isin(empresas_entrenamiento)
    X = df_model.drop(columns=['P_E']); y = df_model['P_E']
    X_train, y_train = X[mask_entrenamiento].copy(), y[mask_entrenamiento].copy()

    if n_empresas_prueba > 0 and len(empresas_prueba) > 0:
        mask_prueba = df_model['Empresa'].isin(empresas_prueba)
        X_test, y_test = X[mask_prueba].copy(), y[mask_prueba].copy()
        # Si X_test es vacío pero se esperaban datos de prueba (mask_prueba no vacía), inicializarlo vacío con columnas correctas
        if X_test.empty and not mask_prueba.empty() and not X_train.empty : 
             X_test = pd.DataFrame(columns=X_train.columns)
             y_test = pd.Series(dtype=y_train.dtype)

    else: # No hay empresas de prueba o X_train está vacío
        X_test_cols = X_train.columns if not X_train.empty else (X.columns if not X.empty else [])
        y_test_dtype = y_train.dtype if not y_train.empty else (y.dtype if not y.empty else float)
        X_test, y_test = pd.DataFrame(columns=X_test_cols), pd.Series(dtype=y_test_dtype)


    if X_train.empty and n_total_empresas > 0 : # Si X_train está vacío pero había datos, es un error
        print("ERROR train_test_split: X_train está vacío después del split.")
        return None, None, None, None
        
    return X_train, y_train, X_test, y_test


def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    
    mae = mean_absolute_error(y_test_np, y_pred_np)
    rmse_val = rmse(y_test_np, y_pred_np) # rmse ya maneja NaNs/Infs
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

def sort_by_timestep(X_df, y_series):
    X, y = X_df.copy(), y_series.copy()
    if 'time_step' not in X.columns: return X, y
    if not isinstance(y, pd.Series) or not X.index.equals(y.index): # Asegurar que los índices coincidan
        if len(y) == len(X): y = pd.Series(np.asarray(y), index=X.index, name=getattr(y, 'name', 'target'))
        else: print("Error sort_by_timestep: y no alineable con X."); return X, y
    X_sorted = X.sort_values('time_step'); y_sorted = y.loc[X_sorted.index]
    return X_sorted, y_sorted

# ================================================================================
# BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON CARACTERÍSTICAS REZAGADAS '_lag1')
# ================================================================================

# ASUNCIONES ANTES DE CORRER ESTE BLOQUE:
# 1. `dfs_shifted`: Diccionario. Key=`metodo`. Value (`df_lag1_original_con_todo`) es un DataFrame que contiene:
#      - IDs: 'Empresa', 'Fecha'.
#      - Target: 'P_E'.
#      - COLUMNAS BASE ORIGINALES del método.
#      - COLUMNAS REZAGADAS `_lag1` (el número de estas depende de filtros previos como `features_kept` en BLOQUE 8).
#    El número de columnas `_lag1` en `dfs_shifted[metodo]` determinará las features para el modelo de ESE `metodo`.

results = []
start_time_total = time.time()
use_log_transform = False
test_size_ratio = 0.20

for metodo, df_lag1_original_con_todo in dfs_shifted.items():
    start_time_config = time.time()
    print("\n" + "="*80)
    print(f"Procesando Método: {metodo} {'(CON Log Transform)' if use_log_transform else '(SIN Log Transform)'} (SOLO LAGS COMO PREDICTORES)")
    print("="*80 + "\n")

    if df_lag1_original_con_todo.empty:
        print(f"  ⇨ ADVERTENCIA: df_lag1_original_con_todo para el método {metodo} está vacío. Omitiendo.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': [], 'n_features_modelo': 0, 'Error': 'Input DataFrame for method is empty'})
        continue

    df_temp_for_method = df_lag1_original_con_todo.copy()

    if 'time_step' not in df_temp_for_method.columns:
        df_temp_for_method.sort_values(['Empresa', 'Fecha'], inplace=True)
        df_temp_for_method['time_step'] = df_temp_for_method.groupby('Empresa').cumcount()

    predictor_cols_solo_lag1 = sorted([c for c in df_temp_for_method.columns if c.endswith('_lag1')])
    
    if not predictor_cols_solo_lag1:
        print(f"  ⇨ ADVERTENCIA: No se encontraron columnas _lag1 en dfs_shifted['{metodo}']. Omitiendo método {metodo}.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': [], 'n_features_modelo': 0, 'Error': f'No lag features found in dfs_shifted for {metodo}'})
        continue
    
    columnas_base_que_generaron_estos_lags = sorted(list(set(lag_col.replace('_lag1', '') for lag_col in predictor_cols_solo_lag1)))
    print(f"  ⇨ Método {metodo}: Se usarán {len(predictor_cols_solo_lag1)} predictores (_lag1).")
    print(f"    Estos lags fueron generados a partir de {len(columnas_base_que_generaron_estos_lags)} columnas base: {columnas_base_que_generaron_estos_lags[:5]}...")

    cols_for_df_model = ['Empresa', 'Fecha', 'time_step', 'P_E'] + predictor_cols_solo_lag1
    final_cols_for_df_model = sorted(list(set(c for c in cols_for_df_model if c in df_temp_for_method.columns))) # Unicas y existentes
    
    essential_check = ['Empresa', 'Fecha', 'time_step', 'P_E']
    if not all(ec in final_cols_for_df_model for ec in essential_check):
        missing_ess_cols = [ec for ec in essential_check if ec not in final_cols_for_df_model]
        print(f"  ⇨ ERROR: Faltan columnas esenciales ({missing_ess_cols}) para construir df_model para {metodo}. Omitiendo.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags, 'n_features_modelo': len(predictor_cols_solo_lag1), 'Error': f'Missing essential cols for df_model: {missing_ess_cols}'})
        continue
        
    df_model = df_temp_for_method[final_cols_for_df_model].copy()

    n_total_empresas_metodo = df_model['Empresa'].nunique()
    n_empresas_test_metodo = 0
    if n_total_empresas_metodo > 1:
        n_empresas_test_metodo = max(1, int(round(n_total_empresas_metodo * test_size_ratio)))
        if n_total_empresas_metodo - n_empresas_test_metodo < 1 : n_empresas_test_metodo = n_total_empresas_metodo - 1
    
    X_train, y_train, X_test, y_test = train_test_split_by_empresa(df_model, n_empresas_test_metodo)
    
    if X_train is None or X_train.empty:
        print(f"  Split train/test fallido o X_train vacío para {metodo}. Omitiendo método.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags, 'n_features_modelo': len(predictor_cols_solo_lag1), 'Error': 'Train/Test split failed or X_train empty'})
        continue
    
    print(f"  ⇨ Shapes post-split: X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape if X_test is not None else 'None'}, y_test: {y_test.shape if y_test is not None else 'None'}")

    y_train_transformed = y_train.copy()
    if use_log_transform:
        y_numeric = pd.to_numeric(y_train, errors='coerce')
        median_y = y_numeric.median() if not y_numeric.isnull().all() else 0.0
        y_numeric = y_numeric.fillna(median_y) # Rellenar NaNs antes de chequear negativos
        y_numeric.loc[y_numeric < 0] = 0 # Asignar 0 a negativos para log1p
        y_train_transformed = np.log1p(y_numeric)
        median_yt = y_train_transformed.median() if not y_train_transformed.isnull().all() else 0.0
        y_train_transformed = y_train_transformed.fillna(median_yt)


    X_train_s, y_train_s = sort_by_timestep(X_train, y_train_transformed)
    X_test_s, y_test_s_original = pd.DataFrame(columns=X_train.columns), pd.Series(dtype=y_train.dtype) # Inicializar
    if X_test is not None and not X_test.empty:
        X_test_s, y_test_s_original = sort_by_timestep(X_test, y_test)

    param_grid_rf   = {'model__max_depth':[10, 20, None],'model__min_samples_split':[5, 10],'model__n_estimators':[100, 200]}
    param_grid_xgb  = {'model__n_estimators':[100, 200],'model__max_depth':[3, 5, 7],'model__learning_rate':[0.1, 0.05]}
    param_grid_lgbm = {'model__n_estimators':[100, 200],'model__learning_rate':[0.1, 0.05],'model__max_depth':[3, 5, 7],'model__num_leaves':[15, 31]}
    param_grid_cb   = {'model__iterations':[100, 200],'model__learning_rate':[0.1, 0.05],'model__depth':[3, 5, 7],'model__l2_leaf_reg':[1, 3]}

    scenarios = {
        'RF_Simple':  {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False},
        'RF_HP':      {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_rf},
        'XGB_Simple': {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False},
        'XGB_HP':     {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_xgb},
        'LGBM_Simple':{'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': False, 'vt': False},
        'LGBM_HP':    {'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': True,  'vt': False, 'grid': param_grid_lgbm},
        'CB_Simple':  {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': False, 'vt': False},
        'CB_HP':      {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': True,  'vt': False, 'grid': param_grid_cb},
    }

    for escenario_nombre, escenario_cfg in scenarios.items():
        tiempo_esc_inicio = time.time()
        print(f"\n  --- Escenario: {escenario_nombre} para Método: {metodo} ---")
        current_base_model, perform_hp_search, use_temporal_cv = escenario_cfg['model'], escenario_cfg['hp'], escenario_cfg['vt']
        X_fit_data, y_fit_data = (X_train_s, y_train_s) if use_temporal_cv else (X_train, y_train_transformed)
        X_eval_data, y_eval_data_original = (X_test_s, y_test_s_original) if use_temporal_cv else (X_test, y_test)

        pipeline_obj = Pipeline([('drop_ids', DropColumns(columns=['Empresa', 'Fecha', 'time_step'])), ('model', current_base_model)])
        final_fitted_model, hp_best_params, error_msg_escenario = None, None, None

        try:
            if perform_hp_search:
                cv_strategy, cv_fit_params = None, {}
                n_groups_cv = X_fit_data['Empresa'].nunique() if 'Empresa' in X_fit_data.columns else 0

                can_do_cv = True
                if use_temporal_cv and n_groups_cv >= 2:
                    X_cv_input = X_fit_data.sort_values(['Empresa', 'time_step']); y_cv_input = y_fit_data.loc[X_cv_input.index]
                    groups_cv = X_cv_input['Empresa']
                    n_splits = min(4, n_groups_cv); n_splits = max(2, n_splits) # GroupKFold min 2
                    if len(X_cv_input) < n_splits: can_do_cv = False; print(f"    Pocas muestras ({len(X_cv_input)}) para {n_splits} splits en GroupKFold.")
                    else: cv_strategy = GroupKFold(n_splits=n_splits); cv_fit_params = {'groups': groups_cv}; print(f"    Usando GroupKFold CV con {n_splits} splits.")
                elif len(X_fit_data) >= 2 : # KFold estándar
                    kfold_splits = min(3, len(X_fit_data)); kfold_splits = max(2, kfold_splits) # KFold min 2
                    if len(X_fit_data) < kfold_splits : can_do_cv = False; print(f"    Pocas muestras ({len(X_fit_data)}) para {kfold_splits} splits en KFold.")
                    else: cv_strategy = KFold(n_splits=kfold_splits, shuffle=True, random_state=42); print(f"    Usando KFold CV con {kfold_splits} splits.")
                else: can_do_cv = False; print(f"    No hay suficientes muestras ({len(X_fit_data)}) para CV. Omitiendo HP search.")
                
                if can_do_cv:
                    gs = GridSearchCV(pipeline_obj, escenario_cfg['grid'], scoring=rmse_scorer, cv=cv_strategy, n_jobs=-1, refit=True, error_score='raise', verbose=0)
                    gs.fit(X_fit_data, y_fit_data, **cv_fit_params)
                    final_fitted_model, hp_best_params = gs.best_estimator_, gs.best_params_
                    print(f"    Mejores parámetros: {hp_best_params}")
                else: perform_hp_search = False # Forzar no HP search y fit simple
            
            if not perform_hp_search or not final_fitted_model: # Si HP search se omitió o falló, fit simple
                pipeline_obj.fit(X_fit_data, y_fit_data)
                final_fitted_model = pipeline_obj
        except Exception as e: error_msg_escenario = str(e); print(f"    ERROR entrenamiento/HP search {escenario_nombre}: {e}")

        eval_metrics, model_n_features = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}, "N/A"
        if final_fitted_model:
            try:
                model_estimator_step = final_fitted_model.steps[-1][1]
                model_n_features = getattr(model_estimator_step, 'n_features_in_', 'N/A')
                print(f"    >> Estimador final '{type(model_estimator_step).__name__}' vio {model_n_features} features.")
                if isinstance(model_n_features, int) and model_n_features != len(predictor_cols_solo_lag1):
                     print(f"       ¡ADVERTENCIA! n_features ({model_n_features}) no coincide con num. de lags ({len(predictor_cols_solo_lag1)}).")

                if X_eval_data is not None and not X_eval_data.empty and y_eval_data_original is not None and not y_eval_data_original.empty:
                    pred_transformed = final_fitted_model.predict(X_eval_data)
                    pred_original_scale = np.expm1(pred_transformed) if use_log_transform else pred_transformed
                    if use_log_transform: pred_original_scale[~np.isfinite(pred_original_scale)] = np.nan
                    eval_metrics = evaluate_model(y_eval_data_original, pred_original_scale)
                    print(f"    Métricas Test {escenario_nombre}: RMSE={eval_metrics['RMSE']:.4f}, MAE={eval_metrics['MAE']:.4f}, R2={eval_metrics['R2']:.4f}")
                else: print("    No hay datos de X_eval/y_eval para evaluar.")
            except Exception as e_eval: error_msg_escenario = (error_msg_escenario + f" | EvalError: {str(e_eval)}") if error_msg_escenario else f"EvalError: {str(e_eval)}"; print(f"    ERROR evaluación {escenario_nombre}: {e_eval}")
        
        results.append({
            'Modelo': escenario_nombre.split('_')[0], 'Validacion_Temp': use_temporal_cv,
            'Busqueda_HP': perform_hp_search and escenario_cfg['hp'], 'Config_Key': metodo, 'Metodo': metodo,
            'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': eval_metrics['RMSE'],
            'MAE': eval_metrics['MAE'], 'R2': eval_metrics['R2'], 'Best_Params': hp_best_params,
            'pipeline': final_fitted_model, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags,
            'n_features_modelo': model_n_features, 'Error': error_msg_escenario})
        print(f"    Tiempo escenario {escenario_nombre}: {time.time() - tiempo_esc_inicio:.2f}s")
    print(f"\nTiempo total Método {metodo}: {time.time() - start_time_config:.2f}s")
print(f"\n\nTiempo total ejecución experimentos: {(time.time() - start_time_total)/60:.2f} minutos")

# ================================================================================
# CREAR TABLA DE RESULTADOS Y GUARDAR MEJOR MODELO (PAQUETE COMPLETO)
# ================================================================================
if results:
    df_results_all_models = pd.DataFrame(results)
    print("\n" + "="*80 + "\n--- Resultados Finales (Modelos Entrenados SOLO CON LAGS) ---\n" + f"Total resultados: {len(df_results_all_models)}\n")
    cols_display_final = ['Metodo', 'Modelo', 'Busqueda_HP', 'RMSE', 'MAE', 'R2', 'n_features_modelo', 'columnas_base_usadas', 'Best_Params', 'Error']
    cols_to_show_final = [col for col in cols_display_final if col in df_results_all_models.columns]
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:.4f}'.format):
        print(df_results_all_models.sort_values(by='RMSE', ascending=True, na_position='last')[cols_to_show_final].to_string(index=False))

    valid_results_for_best = df_results_all_models[df_results_all_models['RMSE'].notna() & df_results_all_models['pipeline'].notna() & df_results_all_models['Error'].isna()].copy()
    if not valid_results_for_best.empty:
        best_model_idx = valid_results_for_best['RMSE'].idxmin()
        best_model_full_info = valid_results_for_best.loc[best_model_idx]
        
        model_package_to_save = {
            'pipeline': best_model_full_info['pipeline'],
            'metodo_features': best_model_full_info['Metodo'], # Clave corregida
            'log_transform_target': best_model_full_info['Log_Transform'],
            'columnas_base_para_lags': best_model_full_info['columnas_base_usadas'],
            'n_features_in_estimator': best_model_full_info['n_features_modelo']}
        
        filename_best_model_package = f"best_model_package_{best_model_full_info['Metodo']}.joblib"
        joblib.dump(model_package_to_save, filename_best_model_package)
        
        print("\n" + "="*80 + "\nPAQUETE DEL MEJOR MODELO (SOLO LAGS) SELECCIONADO Y GUARDADO:\n" +
              f"  Archivo: {filename_best_model_package}\n" +
              f"  Método de Features Original: {best_model_full_info['Metodo']}\n" +
              f"  Columnas Base (de los lags usados): {len(best_model_full_info['columnas_base_usadas'])} {str(best_model_full_info['columnas_base_usadas'][:10]) + ('...' if len(best_model_full_info['columnas_base_usadas']) > 10 else '')}\n" +
              f"  Num. Features para Estimador (lags): {best_model_full_info['n_features_modelo']}\n" +
              f"  Transformación Log Target: {best_model_full_info['Log_Transform']}\n" +
              f"  RMSE en Test: {best_model_full_info['RMSE']:.4f}\n" +
              f"  MAE en Test: {best_model_full_info['MAE']:.4f}\n" +
              f"  R2 en Test: {best_model_full_info['R2']:.4f}\n" +
              f"  Hiperparámetros: {best_model_full_info['Best_Params']}\n" +
              f"  Pipeline: {best_model_full_info['pipeline']}\n" + "="*80)
    else: print("\nNo se encontraron resultados válidos para seleccionar y guardar el mejor modelo.")
else: print("\nNo se generaron resultados en los experimentos.")

In [34]:
df_mejores_por_tipo_global.sort_values(by=["R2"], ascending=False)

,Metodo_Features,Lag_Config,Log_Transform,Modelo_Tipo_Base,RMSE,MAE,R2,n_features_modelo,Best_Params
7,spline_mediana,_lag1_solo_pred,False,CB,28.041951,8.545004,0.217421,14,"{'model__depth': 7, 'model__iterations': 200, ..."
15,spline_promedio,_lag1_solo_pred,False,CB,28.071185,8.413305,0.215789,14,"{'model__depth': 7, 'model__iterations': 200, ..."
10,lineal_promedio,_lag1_solo_pred,False,LGBM,28.238646,8.895504,0.206404,14,None
11,lineal_promedio,_lag1_solo_pred,False,CB,28.261243,8.409971,0.205134,14,"{'model__depth': 7, 'model__iterations': 200, ..."
14,spline_promedio,_lag1_solo_pred,False,LGBM,28.459206,9.055426,0.193959,14,None
6,spline_mediana,_lag1_solo_pred,False,LGBM,28.524839,9.036169,0.190237,14,None
3,lineal_mediana,_lag1_solo_pred,False,CB,28.626233,8.491030,0.184470,14,"{'model__depth': 7, 'model__iterations': 200, ..."
2,lineal_mediana,_lag1_solo_pred,False,LGBM,28.673729,9.261537,0.181762,14,None
0,lineal_mediana,_lag1_solo_pred,False,RF,29.852881,8.631678,0.113081,14,None
8,lineal_promedio,_lag1_solo_pred,False,RF,30.052901,8.663228,0.101156,14,None


In [ ]:
# Celda: BLOQUE DE PREDICCIÓN PARA DLOCAL (CON P_E_lagN, MODELOS INDIVIDUALES - GRÁFICOS Y MÉTRICAS)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

# --- Definiciones de Clases y Funciones Auxiliares ---
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None): self.columns = columns if columns is not None else []
    def fit(self, X, y=None): return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame): return X
        return X.copy().drop(columns=self.columns, errors='ignore')

def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    mae = mean_absolute_error(y_test_np, y_pred_np); rmse_val = rmse(y_test_np, y_pred_np)
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================================================================
# BLOQUE DE PREDICCIÓN PARA DLOCAL (CON P_E_lagN, MODELOS INDIVIDUALES - GRÁFICOS Y MÉTRICAS)
# =====================================================================================

# --- 0. CONFIGURACIÓN Y CARGA DEL PAQUETE DEL MODELO ---
# ASUME: df_dlocal, listas_columnas están definidos.
#        El paquete fue entrenado incluyendo P_E_lagN como feature si se desea usar aquí.

nombre_paquete_a_cargar = "paquete_mejores_por_tipo_lineal_mediana__lag1_solo_pred.joblib" # <--- AJUSTA ESTE NOMBRE

print(f"Cargando paquete de modelos desde: {nombre_paquete_a_cargar}")
model_package = None
try:
    ruta_paquete = os.path.join(os.getcwd(), nombre_paquete_a_cargar)
    if not os.path.exists(ruta_paquete):
        raise FileNotFoundError(f"El archivo del paquete '{ruta_paquete}' no existe.")
    model_package = joblib.load(ruta_paquete)
    print("Paquete de modelos cargado exitosamente.")

    required_keys = ['metodo_features', 'lag_config_usada', 'log_transform_target', 'mejores_modelos_info_por_tipo']
    if not isinstance(model_package, dict) or not all(key in model_package for key in required_keys):
        raise ValueError("El archivo cargado no es un paquete de modelos válido o le faltan claves esenciales.")

    metodo_features_paquete = model_package['metodo_features']
    lag_config_paquete = model_package['lag_config_usada'] # ej. '_lag1_solo_pred', 'lag1_lag3_solo_pred'
    log_transform_paquete = model_package['log_transform_target']
    # 'columnas_base_para_lags' del paquete DEBE incluir 'P_E' si P_E_lagN fue una feature en el entrenamiento.
    columnas_base_paquete = model_package.get('columnas_base_para_lags')
    mejores_modelos_info = model_package['mejores_modelos_info_por_tipo']

    if not columnas_base_paquete:
        if 'listas_columnas' in locals() and metodo_features_paquete in listas_columnas:
            columnas_base_paquete = listas_columnas[metodo_features_paquete]
            # Si P_E_lagN se usó y 'P_E' no está en listas_columnas[metodo_features_paquete], hay que añadirlo:
            if 'P_E' not in columnas_base_paquete and 'P_E' in df_dlocal.columns: # Asumiendo que P_E_lagN se deriva de P_E
                 print("Añadiendo 'P_E' a columnas_base_paquete para generar su lag.")
                 columnas_base_paquete.append('P_E')
                 columnas_base_paquete = sorted(list(set(columnas_base_paquete)))
        else: raise ValueError("No se pudieron determinar las 'columnas_base_para_lags'.")
    
    print(f"  Información del Paquete: Método={metodo_features_paquete}, LagConfig={lag_config_paquete}, LogT={log_transform_paquete}")
    print(f"    Columnas Base (del paquete/listas_columnas, usadas para generar lags): {len(columnas_base_paquete)} {columnas_base_paquete[:5]}...")
    if 'P_E' in columnas_base_paquete:
        print("    'P_E' está en columnas_base_paquete, se generará P_E_lagN.")
    else:
        print("    ADVERTENCIA: 'P_E' NO está en columnas_base_paquete. NO se generará P_E_lagN como feature para DLocal.")


except FileNotFoundError: raise FileNotFoundError(f"ERROR FATAL: No se encontró el archivo del paquete '{nombre_paquete_a_cargar}'.")
except Exception as e_load_pack: raise RuntimeError(f"Error fatal al cargar o procesar el paquete: {e_load_pack}")

if 'df_dlocal' not in locals() or not isinstance(df_dlocal, pd.DataFrame) or df_dlocal.empty:
     raise NameError("'df_dlocal' no definido o vacío.")

# Determinar qué lags generar para DLocal basado en lag_config_paquete
lags_a_generar_dlocal = []
if 'lag1' in lag_config_paquete: lags_a_generar_dlocal.append(1)
if 'lag2' in lag_config_paquete: lags_a_generar_dlocal.append(2) # Por si acaso
if 'lag3' in lag_config_paquete: lags_a_generar_dlocal.append(3)
if not lags_a_generar_dlocal: # Fallback si el string no es claro
    if "lag1" in lag_config_paquete.lower(): lags_a_generar_dlocal.append(1)
    if "lag3" in lag_config_paquete.lower(): lags_a_generar_dlocal.append(3)
    if not lags_a_generar_dlocal:
        raise ValueError(f"No se pudo determinar qué lags numéricos generar a partir de lag_config_paquete: '{lag_config_paquete}'")
print(f"Se generarán lags {lags_a_generar_dlocal} para DLocal.")


# --- 1. PREPARAR df_dlocal ---
print("\n" + "="*80 + f"\nPreparando datos de DLocal (Método: {metodo_features_paquete}, Lags: {lags_a_generar_dlocal})" + "\n" + "="*80)
df_dlocal_procesado_base = df_dlocal.copy()

# 1.1 Conversión Numérica (para todas las columnas base, incluyendo P_E si está en columnas_base_paquete)
cols_convertir_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha']] # No excluir P_E aquí si es una base para lag
for col in cols_convertir_dlocal:
    if not pd.api.types.is_numeric_dtype(df_dlocal_procesado_base[col]):
        df_dlocal_procesado_base[col] = pd.to_numeric(df_dlocal_procesado_base[col], errors='coerce')

# 1.2 Imputación Pre-Lag (para todas las columnas base, incluyendo P_E si está en columnas_base_paquete)
cols_imputar_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha']] # No excluir P_E aquí
if cols_imputar_dlocal:
    df_dlocal_procesado_base.sort_values(['Empresa', 'Fecha'], inplace=True)
    if df_dlocal_procesado_base['Empresa'].nunique() > 1:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base.groupby('Empresa')[cols_imputar_dlocal].ffill()
    else:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].ffill()
    df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].fillna(0)

# 1.3 Generar Lags
# features_para_generar_lags AHORA INCLUIRÁ 'P_E' si 'P_E' está en 'columnas_base_paquete'
features_para_generar_lags_dlocal = [c for c in columnas_base_paquete if c in df_dlocal_procesado_base.columns]
# Asegurar que las columnas de ID y el target P_E (actual) estén para el merge/uso posterior
cols_necesarias_para_base_df = ['Empresa', 'Fecha']
if 'P_E' in df_dlocal_procesado_base.columns: # El P_E actual
    cols_necesarias_para_base_df.append('P_E')
# Añadir las features para las que se generarán lags, evitando duplicados
for f_lag in features_para_generar_lags_dlocal:
    if f_lag not in cols_necesarias_para_base_df:
        cols_necesarias_para_base_df.append(f_lag)

df_dlocal_con_todos_lags = df_dlocal_procesado_base[cols_necesarias_para_base_df].copy()
all_generated_lag_cols_dlocal = []

print(f"Generando lags para DLocal para las features: {features_para_generar_lags_dlocal}")
for lag_val in lags_a_generar_dlocal:
    for col_base_para_lag in features_para_generar_lags_dlocal: # Iterar sobre las que SÍ se van a laggear
        lag_col_name = f"{col_base_para_lag}_lag{lag_val}"
        df_dlocal_con_todos_lags[lag_col_name] = df_dlocal_con_todos_lags.groupby('Empresa')[col_base_para_lag].shift(lag_val)
        all_generated_lag_cols_dlocal.append(lag_col_name)
all_generated_lag_cols_dlocal = sorted(list(set(all_generated_lag_cols_dlocal)))
print(f"Se generaron {len(all_generated_lag_cols_dlocal)} columnas de lag para DLocal: {all_generated_lag_cols_dlocal[:5]}...")
if f'P_E_lag{lags_a_generar_dlocal[0] if lags_a_generar_dlocal else ""}' in all_generated_lag_cols_dlocal:
    print("    ¡Confirmado! P_E_lagN está entre los lags generados.")
else:
    print("    ADVERTENCIA: P_E_lagN NO está entre los lags generados. Verifica 'columnas_base_paquete'.")


# 1.4 Aplicar dropna
df_dlocal_final_features_con_ids = df_dlocal_con_todos_lags.dropna(subset=all_generated_lag_cols_dlocal).copy()
if df_dlocal_final_features_con_ids.empty: raise ValueError("DLocal vacío post lags y dropna.")
print(f"DataFrame DLocal procesado y listo (con IDs y lags): {df_dlocal_final_features_con_ids.shape}")

# --- 2. ITERAR, PREDECIR, EVALUAR INDIVIDUALMENTE Y GRAFICAR ---
# ... (El resto del bloque (sección 2 y 3) es idéntico al que te proporcioné antes
#      que itera sobre mejores_modelos_info, predice, calcula métricas individuales,
#      las guarda en lista_metricas_individuales_dlocal, y grafica individualmente) ...
# ... (Al final, la sección 3 imprime la tabla df_metricas_final_dlocal) ...
print("\n" + "="*80 + "\nIterando sobre modelos por tipo para predicción, evaluación y graficación individual en DLocal" + "\n" + "="*80)
lista_metricas_individuales_dlocal = []

for tipo_modelo_actual, model_info_actual in mejores_modelos_info.items():
    print(f"\n--- Procesando Modelo Tipo: {tipo_modelo_actual} ---")
    if model_info_actual is None or 'pipeline_object' not in model_info_actual or model_info_actual['pipeline_object'] is None:
        lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    pipeline_cargado_actual = model_info_actual['pipeline_object']
    estimador_real_actual = pipeline_cargado_actual
    if hasattr(pipeline_cargado_actual, 'steps'): estimador_real_actual = pipeline_cargado_actual.steps[-1][1]
    features_esperadas_por_modelo_actual = []
    if hasattr(estimador_real_actual, 'feature_names_in_'): features_esperadas_por_modelo_actual = list(estimador_real_actual.feature_names_in_)
    elif hasattr(estimador_real_actual, 'n_features_in_'):
        if estimador_real_actual.n_features_in_ == len(all_generated_lag_cols_dlocal): features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
        elif (tipo_modelo_actual == 'CB' and estimador_real_actual.n_features_in_ == 0 and len(all_generated_lag_cols_dlocal) > 0) or \
             (estimador_real_actual.n_features_in_ != len(all_generated_lag_cols_dlocal) and len(all_generated_lag_cols_dlocal) > 0) :
            features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
        else: lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    else: features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
    if not features_esperadas_por_modelo_actual: lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    missing_features_in_dlocal = [f for f in features_esperadas_por_modelo_actual if f not in df_dlocal_final_features_con_ids.columns]
    if missing_features_in_dlocal: lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    X_dlocal_input_this_model = df_dlocal_final_features_con_ids[features_esperadas_por_modelo_actual].copy()
    df_dlocal_eval_data_this_model = df_dlocal_final_features_con_ids[['Fecha', 'P_E']].loc[X_dlocal_input_this_model.index].copy()
    if X_dlocal_input_this_model.empty: lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    n_rows_pred_this_model = len(X_dlocal_input_this_model); fechas_p_individual, preds_p_individual, reales_p_individual = [], [], []
    if n_rows_pred_this_model > 0:
        if n_rows_pred_this_model > 1:
            for i in range(1, n_rows_pred_this_model):
                X_pred_row_loop = X_dlocal_input_this_model.iloc[[i-1]]
                try:
                    pred_raw_loop = pipeline_cargado_actual.predict(X_pred_row_loop)[0]
                    pred_orig_loop = np.expm1(pred_raw_loop) if log_transform_paquete else pred_raw_loop
                    if log_transform_paquete and not np.isfinite(pred_orig_loop): pred_orig_loop = np.nan
                except Exception: pred_orig_loop = np.nan
                preds_p_individual.append(pred_orig_loop); fechas_p_individual.append(df_dlocal_eval_data_this_model.iloc[i]['Fecha']); reales_p_individual.append(df_dlocal_eval_data_this_model.iloc[i]['P_E'])
        X_forecast_row_loop = X_dlocal_input_this_model.iloc[[-1]]; pronostico_final_individual = np.nan
        try:
            pred_raw_forecast_loop = pipeline_cargado_actual.predict(X_forecast_row_loop)[0]
            pronostico_final_individual = np.expm1(pred_raw_forecast_loop) if log_transform_paquete else pred_raw_forecast_loop
            if log_transform_paquete and not np.isfinite(pronostico_final_individual): pronostico_final_individual = np.nan
        except Exception: pass
        proxima_fecha_individual = "Periodo Siguiente"
        if not df_dlocal_eval_data_this_model.empty and 'Fecha' in df_dlocal_eval_data_this_model.columns and not df_dlocal_eval_data_this_model['Fecha'].empty:
            try: proxima_fecha_individual = df_dlocal_eval_data_this_model['Fecha'].iloc[-1] + pd.DateOffset(months=1)
            except: pass
        df_backtest_individual = pd.DataFrame({'Fecha': fechas_p_individual, 'P_E_Predicho': preds_p_individual, 'P_E_Real': reales_p_individual})
        metricas_individual_actual = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
        if not df_backtest_individual.empty:
            metricas_individual_actual = evaluate_model(df_backtest_individual['P_E_Real'], df_backtest_individual['P_E_Predicho'])
            print(f"  Métricas Backtest ({tipo_modelo_actual}): RMSE={metricas_individual_actual['RMSE']:.4f}, MAE={metricas_individual_actual['MAE']:.4f}, R2={metricas_individual_actual['R2']:.4f}")
        metricas_a_guardar = metricas_individual_actual.copy(); metricas_a_guardar['Modelo_Tipo'] = tipo_modelo_actual
        lista_metricas_individuales_dlocal.append(metricas_a_guardar)
        if not df_backtest_individual.empty:
            plt.figure(figsize=(14, 7))
            plt.plot(df_backtest_individual['Fecha'], df_backtest_individual['P_E_Real'], marker='.', linestyle='-', label='P/E Real DLocal')
            plt.plot(df_backtest_individual['Fecha'], df_backtest_individual['P_E_Predicho'], marker='x', linestyle='--', label=f'P/E Predicho ({tipo_modelo_actual})')
            if isinstance(proxima_fecha_individual, pd.Timestamp) and pd.notna(pronostico_final_individual):
                 plt.scatter([proxima_fecha_individual], [pronostico_final_individual], color='red', label=f'Forecast {tipo_modelo_actual} {proxima_fecha_individual.strftime("%Y-%m")}', zorder=5, s=100)
            plt.title(f'Backtest y Forecast P/E DLocal (Modelo: {tipo_modelo_actual} del Paquete: {metodo_features_paquete}, Lags: {lags_a_generar_dlocal})')
            plt.xlabel('Fecha'); plt.ylabel('P_E Ratio'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
    else:
        lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan})

# --- 3. IMPRIMIR TABLA DE MÉTRICAS INDIVIDUALES (AL FINAL) ---
if lista_metricas_individuales_dlocal:
    df_metricas_final_dlocal = pd.DataFrame(lista_metricas_individuales_dlocal)
    print("\n" + "="*80 + "\nResumen de Métricas Individuales de los Modelos en el Backtest de DLocal" + "\n" + "="*80)
    df_metricas_final_dlocal_sorted = df_metricas_final_dlocal.sort_values(by='RMSE', ascending=True)
    columnas_ordenadas_metricas = ['Modelo_Tipo', 'RMSE', 'MAE', 'R2']
    columnas_existentes_para_print = [col for col in columnas_ordenadas_metricas if col in df_metricas_final_dlocal_sorted.columns]
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000, 'display.float_format', '{:.4f}'.format):
        print(df_metricas_final_dlocal_sorted[columnas_existentes_para_print].to_string(index=False))
else:
    print("\nNo se generaron métricas para DLocal o la lista está vacía.")

print("\n" + "="*80 + "\nProceso de predicción para DLocal completado." + "\n" + "="*80)

#### Con transformación logarítmica

In [ ]:
# Celda: BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON LAGS Y GUARDAR PAQUETE COMPLETO)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib

# --- Modelos ---
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# --- Métricas y Preprocesamiento ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupKFold # BaseCrossValidator no se usa directamente
from sklearn.pipeline import Pipeline

# Ignorar warnings comunes (opcional)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================
# CLASE AUXILIAR DropColumns (Definir una vez)
# =====================================
class DropColumns(BaseEstimator, TransformerMixin):
    """Transformer para eliminar columnas especificadas en un Pipeline."""
    def __init__(self, columns=None):
        self.columns = columns if columns is not None else []
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            return X
        X_transformed = X.copy()
        return X_transformed.drop(columns=self.columns, errors='ignore')

# =====================================
# FUNCIONES AUXILIARES
# =====================================
def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_test_split_by_empresa(df_model_input, n_empresas_prueba):
    df_model = df_model_input.copy()
    if 'Empresa' not in df_model.columns: raise KeyError("'Empresa' necesaria")
    if 'P_E' not in df_model.columns: raise KeyError("'P_E' (target) necesaria")

    empresas_unicas = df_model['Empresa'].unique()
    n_total_empresas = len(empresas_unicas)

    if n_total_empresas == 0: return None, None, None, None
    if n_total_empresas == 1:
        X = df_model.drop(columns=['P_E']); y = df_model['P_E']
        return X.copy(), y.copy(), pd.DataFrame(columns=X.columns), pd.Series(dtype=y.dtype)

    # Asegurar que n_empresas_prueba sea válido (entre 0 y n_total_empresas - 1)
    n_empresas_prueba_valido = max(0, min(n_empresas_prueba, n_total_empresas - 1 if n_total_empresas > 0 else 0))
    if n_empresas_prueba_valido != n_empresas_prueba:
        print(f"  Advertencia train_test_split: n_empresas_prueba ajustado de {n_empresas_prueba} a {n_empresas_prueba_valido}")
    n_empresas_prueba = n_empresas_prueba_valido


    if n_empresas_prueba == 0:
        empresas_entrenamiento = empresas_unicas; empresas_prueba = np.array([])
    else:
        empresas_entrenamiento, empresas_prueba = train_test_split(
            empresas_unicas, test_size=n_empresas_prueba, random_state=42, shuffle=True)

    mask_entrenamiento = df_model['Empresa'].isin(empresas_entrenamiento)
    X = df_model.drop(columns=['P_E']); y = df_model['P_E']
    X_train, y_train = X[mask_entrenamiento].copy(), y[mask_entrenamiento].copy()

    if n_empresas_prueba > 0 and len(empresas_prueba) > 0:
        mask_prueba = df_model['Empresa'].isin(empresas_prueba)
        X_test, y_test = X[mask_prueba].copy(), y[mask_prueba].copy()
        # Si X_test es vacío pero se esperaban datos de prueba (mask_prueba no vacía), inicializarlo vacío con columnas correctas
        if X_test.empty and not mask_prueba.empty() and not X_train.empty : 
             X_test = pd.DataFrame(columns=X_train.columns)
             y_test = pd.Series(dtype=y_train.dtype)

    else: # No hay empresas de prueba o X_train está vacío
        X_test_cols = X_train.columns if not X_train.empty else (X.columns if not X.empty else [])
        y_test_dtype = y_train.dtype if not y_train.empty else (y.dtype if not y.empty else float)
        X_test, y_test = pd.DataFrame(columns=X_test_cols), pd.Series(dtype=y_test_dtype)


    if X_train.empty and n_total_empresas > 0 : # Si X_train está vacío pero había datos, es un error
        print("ERROR train_test_split: X_train está vacío después del split.")
        return None, None, None, None
        
    return X_train, y_train, X_test, y_test


def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    
    mae = mean_absolute_error(y_test_np, y_pred_np)
    rmse_val = rmse(y_test_np, y_pred_np) # rmse ya maneja NaNs/Infs
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

def sort_by_timestep(X_df, y_series):
    X, y = X_df.copy(), y_series.copy()
    if 'time_step' not in X.columns: return X, y
    if not isinstance(y, pd.Series) or not X.index.equals(y.index): # Asegurar que los índices coincidan
        if len(y) == len(X): y = pd.Series(np.asarray(y), index=X.index, name=getattr(y, 'name', 'target'))
        else: print("Error sort_by_timestep: y no alineable con X."); return X, y
    X_sorted = X.sort_values('time_step'); y_sorted = y.loc[X_sorted.index]
    return X_sorted, y_sorted

# ================================================================================
# BLOQUE DE EXPERIMENTOS (ENTRENAR SOLO CON CARACTERÍSTICAS REZAGADAS '_lag1')
# ================================================================================

# ASUNCIONES ANTES DE CORRER ESTE BLOQUE:
# 1. `dfs_shifted`: Diccionario. Key=`metodo`. Value (`df_lag1_original_con_todo`) es un DataFrame que contiene:
#      - IDs: 'Empresa', 'Fecha'.
#      - Target: 'P_E'.
#      - COLUMNAS BASE ORIGINALES del método.
#      - COLUMNAS REZAGADAS `_lag1` (el número de estas depende de filtros previos como `features_kept` en BLOQUE 8).
#    El número de columnas `_lag1` en `dfs_shifted[metodo]` determinará las features para el modelo de ESE `metodo`.

results = []
start_time_total = time.time()
use_log_transform = True
test_size_ratio = 0.20

for metodo, df_lag1_original_con_todo in dfs_shifted.items():
    start_time_config = time.time()
    print("\n" + "="*80)
    print(f"Procesando Método: {metodo} {'(CON Log Transform)' if use_log_transform else '(SIN Log Transform)'} (SOLO LAGS COMO PREDICTORES)")
    print("="*80 + "\n")

    if df_lag1_original_con_todo.empty:
        print(f"  ⇨ ADVERTENCIA: df_lag1_original_con_todo para el método {metodo} está vacío. Omitiendo.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': [], 'n_features_modelo': 0, 'Error': 'Input DataFrame for method is empty'})
        continue

    df_temp_for_method = df_lag1_original_con_todo.copy()

    if 'time_step' not in df_temp_for_method.columns:
        df_temp_for_method.sort_values(['Empresa', 'Fecha'], inplace=True)
        df_temp_for_method['time_step'] = df_temp_for_method.groupby('Empresa').cumcount()

    predictor_cols_solo_lag1 = sorted([c for c in df_temp_for_method.columns if c.endswith('_lag1')])
    
    if not predictor_cols_solo_lag1:
        print(f"  ⇨ ADVERTENCIA: No se encontraron columnas _lag1 en dfs_shifted['{metodo}']. Omitiendo método {metodo}.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': [], 'n_features_modelo': 0, 'Error': f'No lag features found in dfs_shifted for {metodo}'})
        continue
    
    columnas_base_que_generaron_estos_lags = sorted(list(set(lag_col.replace('_lag1', '') for lag_col in predictor_cols_solo_lag1)))
    print(f"  ⇨ Método {metodo}: Se usarán {len(predictor_cols_solo_lag1)} predictores (_lag1).")
    print(f"    Estos lags fueron generados a partir de {len(columnas_base_que_generaron_estos_lags)} columnas base: {columnas_base_que_generaron_estos_lags[:5]}...")

    cols_for_df_model = ['Empresa', 'Fecha', 'time_step', 'P_E'] + predictor_cols_solo_lag1
    final_cols_for_df_model = sorted(list(set(c for c in cols_for_df_model if c in df_temp_for_method.columns))) # Unicas y existentes
    
    essential_check = ['Empresa', 'Fecha', 'time_step', 'P_E']
    if not all(ec in final_cols_for_df_model for ec in essential_check):
        missing_ess_cols = [ec for ec in essential_check if ec not in final_cols_for_df_model]
        print(f"  ⇨ ERROR: Faltan columnas esenciales ({missing_ess_cols}) para construir df_model para {metodo}. Omitiendo.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags, 'n_features_modelo': len(predictor_cols_solo_lag1), 'Error': f'Missing essential cols for df_model: {missing_ess_cols}'})
        continue
        
    df_model = df_temp_for_method[final_cols_for_df_model].copy()

    n_total_empresas_metodo = df_model['Empresa'].nunique()
    n_empresas_test_metodo = 0
    if n_total_empresas_metodo > 1:
        n_empresas_test_metodo = max(1, int(round(n_total_empresas_metodo * test_size_ratio)))
        if n_total_empresas_metodo - n_empresas_test_metodo < 1 : n_empresas_test_metodo = n_total_empresas_metodo - 1
    
    X_train, y_train, X_test, y_test = train_test_split_by_empresa(df_model, n_empresas_test_metodo)
    
    if X_train is None or X_train.empty:
        print(f"  Split train/test fallido o X_train vacío para {metodo}. Omitiendo método.")
        results.append({'Modelo': 'N/A', 'Validacion_Temp': False, 'Busqueda_HP': False, 'Config_Key': metodo, 'Metodo': metodo,
                        'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': np.nan, 'MAE': np.nan,
                        'R2': np.nan, 'Best_Params': None, 'pipeline': None, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags, 'n_features_modelo': len(predictor_cols_solo_lag1), 'Error': 'Train/Test split failed or X_train empty'})
        continue
    
    print(f"  ⇨ Shapes post-split: X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape if X_test is not None else 'None'}, y_test: {y_test.shape if y_test is not None else 'None'}")

    y_train_transformed = y_train.copy()
    if use_log_transform:
        y_numeric = pd.to_numeric(y_train, errors='coerce')
        median_y = y_numeric.median() if not y_numeric.isnull().all() else 0.0
        y_numeric = y_numeric.fillna(median_y) # Rellenar NaNs antes de chequear negativos
        y_numeric.loc[y_numeric < 0] = 0 # Asignar 0 a negativos para log1p
        y_train_transformed = np.log1p(y_numeric)
        median_yt = y_train_transformed.median() if not y_train_transformed.isnull().all() else 0.0
        y_train_transformed = y_train_transformed.fillna(median_yt)


    X_train_s, y_train_s = sort_by_timestep(X_train, y_train_transformed)
    X_test_s, y_test_s_original = pd.DataFrame(columns=X_train.columns), pd.Series(dtype=y_train.dtype) # Inicializar
    if X_test is not None and not X_test.empty:
        X_test_s, y_test_s_original = sort_by_timestep(X_test, y_test)

    param_grid_rf   = {'model__max_depth':[10, 20, None],'model__min_samples_split':[5, 10],'model__n_estimators':[100, 200]}
    param_grid_xgb  = {'model__n_estimators':[100, 200],'model__max_depth':[3, 5, 7],'model__learning_rate':[0.1, 0.05]}
    param_grid_lgbm = {'model__n_estimators':[100, 200],'model__learning_rate':[0.1, 0.05],'model__max_depth':[3, 5, 7],'model__num_leaves':[15, 31]}
    param_grid_cb   = {'model__iterations':[100, 200],'model__learning_rate':[0.1, 0.05],'model__depth':[3, 5, 7],'model__l2_leaf_reg':[1, 3]}

    scenarios = {
        'RF_Simple':  {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False},
        'RF_HP':      {'model': RandomForestRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_rf},
        'XGB_Simple': {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': False, 'vt': False},
        'XGB_HP':     {'model': XGBRegressor(random_state=42, n_jobs=-1), 'hp': True,  'vt': False, 'grid': param_grid_xgb},
        'LGBM_Simple':{'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': False, 'vt': False},
        'LGBM_HP':    {'model': LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1), 'hp': True,  'vt': False, 'grid': param_grid_lgbm},
        'CB_Simple':  {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': False, 'vt': False},
        'CB_HP':      {'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), 'hp': True,  'vt': False, 'grid': param_grid_cb},
    }

    for escenario_nombre, escenario_cfg in scenarios.items():
        tiempo_esc_inicio = time.time()
        print(f"\n  --- Escenario: {escenario_nombre} para Método: {metodo} ---")
        current_base_model, perform_hp_search, use_temporal_cv = escenario_cfg['model'], escenario_cfg['hp'], escenario_cfg['vt']
        X_fit_data, y_fit_data = (X_train_s, y_train_s) if use_temporal_cv else (X_train, y_train_transformed)
        X_eval_data, y_eval_data_original = (X_test_s, y_test_s_original) if use_temporal_cv else (X_test, y_test)

        pipeline_obj = Pipeline([('drop_ids', DropColumns(columns=['Empresa', 'Fecha', 'time_step'])), ('model', current_base_model)])
        final_fitted_model, hp_best_params, error_msg_escenario = None, None, None

        try:
            if perform_hp_search:
                cv_strategy, cv_fit_params = None, {}
                n_groups_cv = X_fit_data['Empresa'].nunique() if 'Empresa' in X_fit_data.columns else 0

                can_do_cv = True
                if use_temporal_cv and n_groups_cv >= 2:
                    X_cv_input = X_fit_data.sort_values(['Empresa', 'time_step']); y_cv_input = y_fit_data.loc[X_cv_input.index]
                    groups_cv = X_cv_input['Empresa']
                    n_splits = min(4, n_groups_cv); n_splits = max(2, n_splits) # GroupKFold min 2
                    if len(X_cv_input) < n_splits: can_do_cv = False; print(f"    Pocas muestras ({len(X_cv_input)}) para {n_splits} splits en GroupKFold.")
                    else: cv_strategy = GroupKFold(n_splits=n_splits); cv_fit_params = {'groups': groups_cv}; print(f"    Usando GroupKFold CV con {n_splits} splits.")
                elif len(X_fit_data) >= 2 : # KFold estándar
                    kfold_splits = min(3, len(X_fit_data)); kfold_splits = max(2, kfold_splits) # KFold min 2
                    if len(X_fit_data) < kfold_splits : can_do_cv = False; print(f"    Pocas muestras ({len(X_fit_data)}) para {kfold_splits} splits en KFold.")
                    else: cv_strategy = KFold(n_splits=kfold_splits, shuffle=True, random_state=42); print(f"    Usando KFold CV con {kfold_splits} splits.")
                else: can_do_cv = False; print(f"    No hay suficientes muestras ({len(X_fit_data)}) para CV. Omitiendo HP search.")
                
                if can_do_cv:
                    gs = GridSearchCV(pipeline_obj, escenario_cfg['grid'], scoring=rmse_scorer, cv=cv_strategy, n_jobs=-1, refit=True, error_score='raise', verbose=0)
                    gs.fit(X_fit_data, y_fit_data, **cv_fit_params)
                    final_fitted_model, hp_best_params = gs.best_estimator_, gs.best_params_
                    print(f"    Mejores parámetros: {hp_best_params}")
                else: perform_hp_search = False # Forzar no HP search y fit simple
            
            if not perform_hp_search or not final_fitted_model: # Si HP search se omitió o falló, fit simple
                pipeline_obj.fit(X_fit_data, y_fit_data)
                final_fitted_model = pipeline_obj
        except Exception as e: error_msg_escenario = str(e); print(f"    ERROR entrenamiento/HP search {escenario_nombre}: {e}")

        eval_metrics, model_n_features = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}, "N/A"
        if final_fitted_model:
            try:
                model_estimator_step = final_fitted_model.steps[-1][1]
                model_n_features = getattr(model_estimator_step, 'n_features_in_', 'N/A')
                print(f"    >> Estimador final '{type(model_estimator_step).__name__}' vio {model_n_features} features.")
                if isinstance(model_n_features, int) and model_n_features != len(predictor_cols_solo_lag1):
                     print(f"       ¡ADVERTENCIA! n_features ({model_n_features}) no coincide con num. de lags ({len(predictor_cols_solo_lag1)}).")

                if X_eval_data is not None and not X_eval_data.empty and y_eval_data_original is not None and not y_eval_data_original.empty:
                    pred_transformed = final_fitted_model.predict(X_eval_data)
                    pred_original_scale = np.expm1(pred_transformed) if use_log_transform else pred_transformed
                    if use_log_transform: pred_original_scale[~np.isfinite(pred_original_scale)] = np.nan
                    eval_metrics = evaluate_model(y_eval_data_original, pred_original_scale)
                    print(f"    Métricas Test {escenario_nombre}: RMSE={eval_metrics['RMSE']:.4f}, MAE={eval_metrics['MAE']:.4f}, R2={eval_metrics['R2']:.4f}")
                else: print("    No hay datos de X_eval/y_eval para evaluar.")
            except Exception as e_eval: error_msg_escenario = (error_msg_escenario + f" | EvalError: {str(e_eval)}") if error_msg_escenario else f"EvalError: {str(e_eval)}"; print(f"    ERROR evaluación {escenario_nombre}: {e_eval}")
        
        results.append({
            'Modelo': escenario_nombre.split('_')[0], 'Validacion_Temp': use_temporal_cv,
            'Busqueda_HP': perform_hp_search and escenario_cfg['hp'], 'Config_Key': metodo, 'Metodo': metodo,
            'Lag_Config': 'lag1_solo_pred', 'Log_Transform': use_log_transform, 'RMSE': eval_metrics['RMSE'],
            'MAE': eval_metrics['MAE'], 'R2': eval_metrics['R2'], 'Best_Params': hp_best_params,
            'pipeline': final_fitted_model, 'columnas_base_usadas': columnas_base_que_generaron_estos_lags,
            'n_features_modelo': model_n_features, 'Error': error_msg_escenario})
        print(f"    Tiempo escenario {escenario_nombre}: {time.time() - tiempo_esc_inicio:.2f}s")
    print(f"\nTiempo total Método {metodo}: {time.time() - start_time_config:.2f}s")
print(f"\n\nTiempo total ejecución experimentos: {(time.time() - start_time_total)/60:.2f} minutos")

# ================================================================================
# CREAR TABLA DE RESULTADOS Y GUARDAR MEJOR MODELO (PAQUETE COMPLETO)
# ================================================================================
if results:
    df_results_all_models = pd.DataFrame(results)
    print("\n" + "="*80 + "\n--- Resultados Finales (Modelos Entrenados SOLO CON LAGS) ---\n" + f"Total resultados: {len(df_results_all_models)}\n")
    cols_display_final = ['Metodo', 'Modelo', 'Busqueda_HP', 'RMSE', 'MAE', 'R2', 'n_features_modelo', 'columnas_base_usadas', 'Best_Params', 'Error']
    cols_to_show_final = [col for col in cols_display_final if col in df_results_all_models.columns]
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:.4f}'.format):
        print(df_results_all_models.sort_values(by='RMSE', ascending=True, na_position='last')[cols_to_show_final].to_string(index=False))

    valid_results_for_best = df_results_all_models[df_results_all_models['RMSE'].notna() & df_results_all_models['pipeline'].notna() & df_results_all_models['Error'].isna()].copy()
    if not valid_results_for_best.empty:
        best_model_idx = valid_results_for_best['RMSE'].idxmin()
        best_model_full_info = valid_results_for_best.loc[best_model_idx]
        
        model_package_to_save = {
            'pipeline': best_model_full_info['pipeline'],
            'metodo_features': best_model_full_info['Metodo'], # Clave corregida
            'log_transform_target': best_model_full_info['Log_Transform'],
            'columnas_base_para_lags': best_model_full_info['columnas_base_usadas'],
            'n_features_in_estimator': best_model_full_info['n_features_modelo']}
        
        filename_best_model_package = f"best_model_package_{best_model_full_info['Metodo']}.joblib"
        joblib.dump(model_package_to_save, filename_best_model_package)
        
        print("\n" + "="*80 + "\nPAQUETE DEL MEJOR MODELO (SOLO LAGS) SELECCIONADO Y GUARDADO:\n" +
              f"  Archivo: {filename_best_model_package}\n" +
              f"  Método de Features Original: {best_model_full_info['Metodo']}\n" +
              f"  Columnas Base (de los lags usados): {len(best_model_full_info['columnas_base_usadas'])} {str(best_model_full_info['columnas_base_usadas'][:10]) + ('...' if len(best_model_full_info['columnas_base_usadas']) > 10 else '')}\n" +
              f"  Num. Features para Estimador (lags): {best_model_full_info['n_features_modelo']}\n" +
              f"  Transformación Log Target: {best_model_full_info['Log_Transform']}\n" +
              f"  RMSE en Test: {best_model_full_info['RMSE']:.4f}\n" +
              f"  MAE en Test: {best_model_full_info['MAE']:.4f}\n" +
              f"  R2 en Test: {best_model_full_info['R2']:.4f}\n" +
              f"  Hiperparámetros: {best_model_full_info['Best_Params']}\n" +
              f"  Pipeline: {best_model_full_info['pipeline']}\n" + "="*80)
    else: print("\nNo se encontraron resultados válidos para seleccionar y guardar el mejor modelo.")
else: print("\nNo se generaron resultados en los experimentos.")

In [ ]:
# Celda: BLOQUE DE PREDICCIÓN PARA DLOCAL (CON P_E_lagN, MODELOS INDIVIDUALES - GRÁFICOS Y MÉTRICAS)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

# --- Definiciones de Clases y Funciones Auxiliares ---
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None): self.columns = columns if columns is not None else []
    def fit(self, X, y=None): return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame): return X
        return X.copy().drop(columns=self.columns, errors='ignore')

def rmse(y_true, y_pred):
    y_true_np, y_pred_np = np.asarray(y_true), np.asarray(y_pred)
    mask = np.isfinite(y_true_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_true_np, y_pred_np = y_true_np[mask], y_pred_np[mask]
    if len(y_true_np) == 0: return np.nan
    return np.sqrt(mean_squared_error(y_true_np, y_pred_np))

def evaluate_model(y_test, y_pred):
    y_test_np, y_pred_np = np.asarray(y_test), np.asarray(y_pred)
    mask = np.isfinite(y_test_np) & np.isfinite(y_pred_np)
    if not np.all(mask): y_test_np, y_pred_np = y_test_np[mask], y_pred_np[mask]
    if len(y_test_np) == 0: return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    mae = mean_absolute_error(y_test_np, y_pred_np); rmse_val = rmse(y_test_np, y_pred_np)
    r2 = np.nan
    if len(y_test_np) > 1 and np.var(y_test_np) > 1e-9:
        try: r2 = r2_score(y_test_np, y_pred_np)
        except ValueError: pass
    elif len(y_test_np) > 0 and mean_squared_error(y_test_np, y_pred_np) < 1e-9: r2 = 1.0
    elif len(y_test_np) > 0: r2 = 0.0
    return {'RMSE': rmse_val, 'MAE': mae, 'R2': r2}

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# =====================================================================================
# BLOQUE DE PREDICCIÓN PARA DLOCAL (CON P_E_lagN, MODELOS INDIVIDUALES - GRÁFICOS Y MÉTRICAS)
# =====================================================================================

# --- 0. CONFIGURACIÓN Y CARGA DEL PAQUETE DEL MODELO ---
# ASUME: df_dlocal, listas_columnas están definidos.
#        El paquete fue entrenado incluyendo P_E_lagN como feature si se desea usar aquí.

nombre_paquete_a_cargar = "paquete_mejores_por_tipo_lineal_mediana__lag1_solo_pred.joblib" # <--- AJUSTA ESTE NOMBRE

print(f"Cargando paquete de modelos desde: {nombre_paquete_a_cargar}")
model_package = None
try:
    ruta_paquete = os.path.join(os.getcwd(), nombre_paquete_a_cargar)
    if not os.path.exists(ruta_paquete):
        raise FileNotFoundError(f"El archivo del paquete '{ruta_paquete}' no existe.")
    model_package = joblib.load(ruta_paquete)
    print("Paquete de modelos cargado exitosamente.")

    required_keys = ['metodo_features', 'lag_config_usada', 'log_transform_target', 'mejores_modelos_info_por_tipo']
    if not isinstance(model_package, dict) or not all(key in model_package for key in required_keys):
        raise ValueError("El archivo cargado no es un paquete de modelos válido o le faltan claves esenciales.")

    metodo_features_paquete = model_package['metodo_features']
    lag_config_paquete = model_package['lag_config_usada'] # ej. '_lag1_solo_pred', 'lag1_lag3_solo_pred'
    log_transform_paquete = model_package['log_transform_target']
    # 'columnas_base_para_lags' del paquete DEBE incluir 'P_E' si P_E_lagN fue una feature en el entrenamiento.
    columnas_base_paquete = model_package.get('columnas_base_para_lags')
    mejores_modelos_info = model_package['mejores_modelos_info_por_tipo']

    if not columnas_base_paquete:
        if 'listas_columnas' in locals() and metodo_features_paquete in listas_columnas:
            columnas_base_paquete = listas_columnas[metodo_features_paquete]
            # Si P_E_lagN se usó y 'P_E' no está en listas_columnas[metodo_features_paquete], hay que añadirlo:
            if 'P_E' not in columnas_base_paquete and 'P_E' in df_dlocal.columns: # Asumiendo que P_E_lagN se deriva de P_E
                 print("Añadiendo 'P_E' a columnas_base_paquete para generar su lag.")
                 columnas_base_paquete.append('P_E')
                 columnas_base_paquete = sorted(list(set(columnas_base_paquete)))
        else: raise ValueError("No se pudieron determinar las 'columnas_base_para_lags'.")
    
    print(f"  Información del Paquete: Método={metodo_features_paquete}, LagConfig={lag_config_paquete}, LogT={log_transform_paquete}")
    print(f"    Columnas Base (del paquete/listas_columnas, usadas para generar lags): {len(columnas_base_paquete)} {columnas_base_paquete[:5]}...")
    if 'P_E' in columnas_base_paquete:
        print("    'P_E' está en columnas_base_paquete, se generará P_E_lagN.")
    else:
        print("    ADVERTENCIA: 'P_E' NO está en columnas_base_paquete. NO se generará P_E_lagN como feature para DLocal.")


except FileNotFoundError: raise FileNotFoundError(f"ERROR FATAL: No se encontró el archivo del paquete '{nombre_paquete_a_cargar}'.")
except Exception as e_load_pack: raise RuntimeError(f"Error fatal al cargar o procesar el paquete: {e_load_pack}")

if 'df_dlocal' not in locals() or not isinstance(df_dlocal, pd.DataFrame) or df_dlocal.empty:
     raise NameError("'df_dlocal' no definido o vacío.")

# Determinar qué lags generar para DLocal basado en lag_config_paquete
lags_a_generar_dlocal = []
if 'lag1' in lag_config_paquete: lags_a_generar_dlocal.append(1)
if 'lag2' in lag_config_paquete: lags_a_generar_dlocal.append(2) # Por si acaso
if 'lag3' in lag_config_paquete: lags_a_generar_dlocal.append(3)
if not lags_a_generar_dlocal: # Fallback si el string no es claro
    if "lag1" in lag_config_paquete.lower(): lags_a_generar_dlocal.append(1)
    if "lag3" in lag_config_paquete.lower(): lags_a_generar_dlocal.append(3)
    if not lags_a_generar_dlocal:
        raise ValueError(f"No se pudo determinar qué lags numéricos generar a partir de lag_config_paquete: '{lag_config_paquete}'")
print(f"Se generarán lags {lags_a_generar_dlocal} para DLocal.")


# --- 1. PREPARAR df_dlocal ---
print("\n" + "="*80 + f"\nPreparando datos de DLocal (Método: {metodo_features_paquete}, Lags: {lags_a_generar_dlocal})" + "\n" + "="*80)
df_dlocal_procesado_base = df_dlocal.copy()

# 1.1 Conversión Numérica (para todas las columnas base, incluyendo P_E si está en columnas_base_paquete)
cols_convertir_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha']] # No excluir P_E aquí si es una base para lag
for col in cols_convertir_dlocal:
    if not pd.api.types.is_numeric_dtype(df_dlocal_procesado_base[col]):
        df_dlocal_procesado_base[col] = pd.to_numeric(df_dlocal_procesado_base[col], errors='coerce')

# 1.2 Imputación Pre-Lag (para todas las columnas base, incluyendo P_E si está en columnas_base_paquete)
cols_imputar_dlocal = [col for col in columnas_base_paquete if col in df_dlocal_procesado_base.columns and col not in ['Empresa', 'Fecha']] # No excluir P_E aquí
if cols_imputar_dlocal:
    df_dlocal_procesado_base.sort_values(['Empresa', 'Fecha'], inplace=True)
    if df_dlocal_procesado_base['Empresa'].nunique() > 1:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base.groupby('Empresa')[cols_imputar_dlocal].ffill()
    else:
        df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].ffill()
    df_dlocal_procesado_base[cols_imputar_dlocal] = df_dlocal_procesado_base[cols_imputar_dlocal].fillna(0)

# 1.3 Generar Lags
# features_para_generar_lags AHORA INCLUIRÁ 'P_E' si 'P_E' está en 'columnas_base_paquete'
features_para_generar_lags_dlocal = [c for c in columnas_base_paquete if c in df_dlocal_procesado_base.columns]
# Asegurar que las columnas de ID y el target P_E (actual) estén para el merge/uso posterior
cols_necesarias_para_base_df = ['Empresa', 'Fecha']
if 'P_E' in df_dlocal_procesado_base.columns: # El P_E actual
    cols_necesarias_para_base_df.append('P_E')
# Añadir las features para las que se generarán lags, evitando duplicados
for f_lag in features_para_generar_lags_dlocal:
    if f_lag not in cols_necesarias_para_base_df:
        cols_necesarias_para_base_df.append(f_lag)

df_dlocal_con_todos_lags = df_dlocal_procesado_base[cols_necesarias_para_base_df].copy()
all_generated_lag_cols_dlocal = []

print(f"Generando lags para DLocal para las features: {features_para_generar_lags_dlocal}")
for lag_val in lags_a_generar_dlocal:
    for col_base_para_lag in features_para_generar_lags_dlocal: # Iterar sobre las que SÍ se van a laggear
        lag_col_name = f"{col_base_para_lag}_lag{lag_val}"
        df_dlocal_con_todos_lags[lag_col_name] = df_dlocal_con_todos_lags.groupby('Empresa')[col_base_para_lag].shift(lag_val)
        all_generated_lag_cols_dlocal.append(lag_col_name)
all_generated_lag_cols_dlocal = sorted(list(set(all_generated_lag_cols_dlocal)))
print(f"Se generaron {len(all_generated_lag_cols_dlocal)} columnas de lag para DLocal: {all_generated_lag_cols_dlocal[:5]}...")
if f'P_E_lag{lags_a_generar_dlocal[0] if lags_a_generar_dlocal else ""}' in all_generated_lag_cols_dlocal:
    print("    ¡Confirmado! P_E_lagN está entre los lags generados.")
else:
    print("    ADVERTENCIA: P_E_lagN NO está entre los lags generados. Verifica 'columnas_base_paquete'.")


# 1.4 Aplicar dropna
df_dlocal_final_features_con_ids = df_dlocal_con_todos_lags.dropna(subset=all_generated_lag_cols_dlocal).copy()
if df_dlocal_final_features_con_ids.empty: raise ValueError("DLocal vacío post lags y dropna.")
print(f"DataFrame DLocal procesado y listo (con IDs y lags): {df_dlocal_final_features_con_ids.shape}")

# --- 2. ITERAR, PREDECIR, EVALUAR INDIVIDUALMENTE Y GRAFICAR ---
# ... (El resto del bloque (sección 2 y 3) es idéntico al que te proporcioné antes
#      que itera sobre mejores_modelos_info, predice, calcula métricas individuales,
#      las guarda en lista_metricas_individuales_dlocal, y grafica individualmente) ...
# ... (Al final, la sección 3 imprime la tabla df_metricas_final_dlocal) ...
print("\n" + "="*80 + "\nIterando sobre modelos por tipo para predicción, evaluación y graficación individual en DLocal" + "\n" + "="*80)
lista_metricas_individuales_dlocal = []

for tipo_modelo_actual, model_info_actual in mejores_modelos_info.items():
    print(f"\n--- Procesando Modelo Tipo: {tipo_modelo_actual} ---")
    if model_info_actual is None or 'pipeline_object' not in model_info_actual or model_info_actual['pipeline_object'] is None:
        lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    pipeline_cargado_actual = model_info_actual['pipeline_object']
    estimador_real_actual = pipeline_cargado_actual
    if hasattr(pipeline_cargado_actual, 'steps'): estimador_real_actual = pipeline_cargado_actual.steps[-1][1]
    features_esperadas_por_modelo_actual = []
    if hasattr(estimador_real_actual, 'feature_names_in_'): features_esperadas_por_modelo_actual = list(estimador_real_actual.feature_names_in_)
    elif hasattr(estimador_real_actual, 'n_features_in_'):
        if estimador_real_actual.n_features_in_ == len(all_generated_lag_cols_dlocal): features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
        elif (tipo_modelo_actual == 'CB' and estimador_real_actual.n_features_in_ == 0 and len(all_generated_lag_cols_dlocal) > 0) or \
             (estimador_real_actual.n_features_in_ != len(all_generated_lag_cols_dlocal) and len(all_generated_lag_cols_dlocal) > 0) :
            features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
        else: lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    else: features_esperadas_por_modelo_actual = all_generated_lag_cols_dlocal
    if not features_esperadas_por_modelo_actual: lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    missing_features_in_dlocal = [f for f in features_esperadas_por_modelo_actual if f not in df_dlocal_final_features_con_ids.columns]
    if missing_features_in_dlocal: lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    X_dlocal_input_this_model = df_dlocal_final_features_con_ids[features_esperadas_por_modelo_actual].copy()
    df_dlocal_eval_data_this_model = df_dlocal_final_features_con_ids[['Fecha', 'P_E']].loc[X_dlocal_input_this_model.index].copy()
    if X_dlocal_input_this_model.empty: lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}); continue
    n_rows_pred_this_model = len(X_dlocal_input_this_model); fechas_p_individual, preds_p_individual, reales_p_individual = [], [], []
    if n_rows_pred_this_model > 0:
        if n_rows_pred_this_model > 1:
            for i in range(1, n_rows_pred_this_model):
                X_pred_row_loop = X_dlocal_input_this_model.iloc[[i-1]]
                try:
                    pred_raw_loop = pipeline_cargado_actual.predict(X_pred_row_loop)[0]
                    pred_orig_loop = np.expm1(pred_raw_loop) if log_transform_paquete else pred_raw_loop
                    if log_transform_paquete and not np.isfinite(pred_orig_loop): pred_orig_loop = np.nan
                except Exception: pred_orig_loop = np.nan
                preds_p_individual.append(pred_orig_loop); fechas_p_individual.append(df_dlocal_eval_data_this_model.iloc[i]['Fecha']); reales_p_individual.append(df_dlocal_eval_data_this_model.iloc[i]['P_E'])
        X_forecast_row_loop = X_dlocal_input_this_model.iloc[[-1]]; pronostico_final_individual = np.nan
        try:
            pred_raw_forecast_loop = pipeline_cargado_actual.predict(X_forecast_row_loop)[0]
            pronostico_final_individual = np.expm1(pred_raw_forecast_loop) if log_transform_paquete else pred_raw_forecast_loop
            if log_transform_paquete and not np.isfinite(pronostico_final_individual): pronostico_final_individual = np.nan
        except Exception: pass
        proxima_fecha_individual = "Periodo Siguiente"
        if not df_dlocal_eval_data_this_model.empty and 'Fecha' in df_dlocal_eval_data_this_model.columns and not df_dlocal_eval_data_this_model['Fecha'].empty:
            try: proxima_fecha_individual = df_dlocal_eval_data_this_model['Fecha'].iloc[-1] + pd.DateOffset(months=1)
            except: pass
        df_backtest_individual = pd.DataFrame({'Fecha': fechas_p_individual, 'P_E_Predicho': preds_p_individual, 'P_E_Real': reales_p_individual})
        metricas_individual_actual = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
        if not df_backtest_individual.empty:
            metricas_individual_actual = evaluate_model(df_backtest_individual['P_E_Real'], df_backtest_individual['P_E_Predicho'])
            print(f"  Métricas Backtest ({tipo_modelo_actual}): RMSE={metricas_individual_actual['RMSE']:.4f}, MAE={metricas_individual_actual['MAE']:.4f}, R2={metricas_individual_actual['R2']:.4f}")
        metricas_a_guardar = metricas_individual_actual.copy(); metricas_a_guardar['Modelo_Tipo'] = tipo_modelo_actual
        lista_metricas_individuales_dlocal.append(metricas_a_guardar)
        if not df_backtest_individual.empty:
            plt.figure(figsize=(14, 7))
            plt.plot(df_backtest_individual['Fecha'], df_backtest_individual['P_E_Real'], marker='.', linestyle='-', label='P/E Real DLocal')
            plt.plot(df_backtest_individual['Fecha'], df_backtest_individual['P_E_Predicho'], marker='x', linestyle='--', label=f'P/E Predicho ({tipo_modelo_actual})')
            if isinstance(proxima_fecha_individual, pd.Timestamp) and pd.notna(pronostico_final_individual):
                 plt.scatter([proxima_fecha_individual], [pronostico_final_individual], color='red', label=f'Forecast {tipo_modelo_actual} {proxima_fecha_individual.strftime("%Y-%m")}', zorder=5, s=100)
            plt.title(f'Backtest y Forecast P/E DLocal (Modelo: {tipo_modelo_actual} del Paquete: {metodo_features_paquete}, Lags: {lags_a_generar_dlocal})')
            plt.xlabel('Fecha'); plt.ylabel('P_E Ratio'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
    else:
        lista_metricas_individuales_dlocal.append({'Modelo_Tipo': tipo_modelo_actual, 'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan})

# --- 3. IMPRIMIR TABLA DE MÉTRICAS INDIVIDUALES (AL FINAL) ---
if lista_metricas_individuales_dlocal:
    df_metricas_final_dlocal = pd.DataFrame(lista_metricas_individuales_dlocal)
    print("\n" + "="*80 + "\nResumen de Métricas Individuales de los Modelos en el Backtest de DLocal" + "\n" + "="*80)
    df_metricas_final_dlocal_sorted = df_metricas_final_dlocal.sort_values(by='RMSE', ascending=True)
    columnas_ordenadas_metricas = ['Modelo_Tipo', 'RMSE', 'MAE', 'R2']
    columnas_existentes_para_print = [col for col in columnas_ordenadas_metricas if col in df_metricas_final_dlocal_sorted.columns]
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000, 'display.float_format', '{:.4f}'.format):
        print(df_metricas_final_dlocal_sorted[columnas_existentes_para_print].to_string(index=False))
else:
    print("\nNo se generaron métricas para DLocal o la lista está vacía.")

print("\n" + "="*80 + "\nProceso de predicción para DLocal completado." + "\n" + "="*80)